In [1]:
import pandas as pd
import numpy as np

print("--- 租金预测任务：开始 ---")

# --- 1. 加载租金训练数据 ---
rent_train_filename = 'ruc_Class25Q2_train_rent.csv' 
try:
    df_train_rent = pd.read_csv(rent_train_filename)
    print(f"成功加载租金训练集 '{rent_train_filename}'，维度: {df_train_rent.shape}")

    # --- 2. 初始检查 ---
    print("\n--- 租金训练集 (df_train_rent) 详细信息 ---")
    df_train_rent.info()

    print("\n--- 租金训练集 (df_train_rent) 前5行预览 ---")
    print(df_train_rent.head())

except FileNotFoundError:
    print(f"错误：未找到文件 '{rent_train_filename}'。请检查文件名。")
except Exception as e:
    print(f"加载文件时出错: {e}")

--- 租金预测任务：开始 ---


C:\Users\wangy\AppData\Local\Temp\ipykernel_27440\815781794.py:9: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train_rent = pd.read_csv(rent_train_filename)


成功加载租金训练集 'ruc_Class25Q2_train_rent.csv'，维度: (98899, 46)

--- 租金训练集 (df_train_rent) 详细信息 ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98899 entries, 0 to 98898
Data columns (total 46 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   城市       98899 non-null  int64  
 1   户型       98898 non-null  object 
 2   装修       25410 non-null  object 
 3   Price    98899 non-null  float64
 4   楼层       98894 non-null  object 
 5   面积       98899 non-null  object 
 6   朝向       98894 non-null  object 
 7   交易时间     98899 non-null  object 
 8   付款方式     80476 non-null  object 
 9   租赁方式     98899 non-null  object 
 10  电梯       98895 non-null  object 
 11  车位       24764 non-null  object 
 12  用水       81159 non-null  object 
 13  用电       81575 non-null  object 
 14  燃气       94317 non-null  object 
 15  采暖       34412 non-null  object 
 16  租期       51966 non-null  object 
 17  配套设施     68448 non-null  object 
 18  lon      98899 non-null  float64


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import re

print("--- 租金预测：步骤 2 & 3 - 基础清洗与分割 ---")

# --- 1. 重命名目标变量 ---
try:
    df_train_rent.rename(columns={'Price': 'Rent'}, inplace=True)
    print("目标变量已重命名为 'Rent'。")
except Exception as e:
    print(f"重命名目标变量时出错: {e}")

# --- 2. 删除无用/泄漏列 ---
# (基于 info 和 head 的初步判断)
rent_cols_to_drop = [
    '客户反馈',         # 数据泄漏
    '物业办公电话',     # 非特征 ID
    '环线位置',         # 高缺失 (>70%)
    '供热费',           # 高缺失 (>70%)
    # --- 考虑删除其他高缺失列 ---
    '装修',             # >70% 缺失
    '车位',             # >75% 缺失 (object 类型，难以填充)
    '采暖',             # >65% 缺失
    '供暖',             # >60% 缺失 (与采暖可能重复?)
    '租期',             # ~50% 缺失 (格式未知，暂不处理)
    # --- 可能冗余的列 (供水/电 和 用水/电) ---
    '供水', '供电'      # 保留 用水, 用电
]
# 确保只删除存在的列
rent_cols_to_drop_present = [col for col in rent_cols_to_drop if col in df_train_rent.columns]
df_rent_cleaned = df_train_rent.drop(columns=rent_cols_to_drop_present).copy()
print(f"已删除 {len(rent_cols_to_drop_present)} 个无用/泄漏/高缺失列。")
print(f"清理后维度: {df_rent_cleaned.shape}")

# --- 3. 清理核心特征 '面积' ---
try:
    df_rent_cleaned['面积'] = df_rent_cleaned['面积'].str.replace('㎡', '').astype(float)
    print("'面积' 列已清理为数值型。")
except Exception as e:
    print(f"清理 '面积' 列时出错: {e}")

# --- 4. 分离 X 和 y 并执行 80/20 分割 ---
try:
    y_rent = df_rent_cleaned['Rent']
    X_rent = df_rent_cleaned.drop(columns=['Rent'])
    
    X_train_rent, X_val_rent, y_train_rent, y_val_rent = train_test_split(
        X_rent, y_rent, test_size=0.2, random_state=111
    )
    
    X_train_rent = X_train_rent.copy()
    X_val_rent = X_val_rent.copy()
    
    # --- 处理 Kaggle 测试集 ---
    # !! 假设 Kaggle 测试集文件和之前一样 !!
    # !! 并且我们也需要对其进行相同的列删除和面积清理 !!
    original_test_filename = 'ruc_Class25Q2_test_rent.csv' # ！！确保文件名正确！！
    df_test = pd.read_csv(original_test_filename)
    if 'ID' in df_test.columns: # 保留 ID 用于最后提交
         test_ids_rent = df_test['ID'] # 存储 ID
    else:
         print("警告：Kaggle 测试集未找到 ID 列。")
         test_ids_rent = None
         
    # 删除与训练集相同的列
    test_cols_to_drop_rent = [col for col in rent_cols_to_drop_present if col in df_test.columns]
    X_test_rent = df_test.drop(columns=test_cols_to_drop_rent, errors='ignore').copy()
    # 清理面积
    if '面积' in X_test_rent.columns:
        X_test_rent['面积'] = X_test_rent['面积'].str.replace('㎡', '').astype(float)
    # 移除 ID (如果存在且不是我们要保留的)
    if 'ID' in X_test_rent.columns and test_ids_rent is not None:
         X_test_rent = X_test_rent.drop(columns=['ID'])
         
    print("80/20 分割完成。Kaggle 测试集已初步处理。")
    print(f"X_train_rent 维度: {X_train_rent.shape}")
    print(f"X_val_rent 维度:   {X_val_rent.shape}")
    print(f"X_test_rent 维度:  {X_test_rent.shape}")
    
except KeyError as e:
     print(f"错误：列 '{e}' 未找到。请检查列名。")
except NameError as e:
     print(f"错误：变量未找到 ({e})。")
except Exception as e:
     print(f"分割或处理测试集时发生错误: {e}")


--- 租金预测：步骤 2 & 3 - 基础清洗与分割 ---
目标变量已重命名为 'Rent'。
已删除 11 个无用/泄漏/高缺失列。
清理后维度: (98899, 35)
'面积' 列已清理为数值型。
80/20 分割完成。Kaggle 测试集已初步处理。
X_train_rent 维度: (79119, 34)
X_val_rent 维度:   (19780, 34)
X_test_rent 维度:  (9773, 34)


In [3]:
# --- 5. 执行第一轮缺失值填充 ---
try:
    numerical_features_rent = X_train_rent.select_dtypes(include=['float64', 'int64']).columns
    categorical_features_rent = X_train_rent.select_dtypes(include=['object']).columns

    print(f"\n开始第一轮缺失值填充 (中位数 / 'Unknown')...")
    for col in numerical_features_rent:
        median_value = X_train_rent[col].median()
        X_train_rent[col] = X_train_rent[col].fillna(median_value)
        X_val_rent[col] = X_val_rent[col].fillna(median_value)
        if col in X_test_rent.columns:
            X_test_rent[col] = X_test_rent[col].fillna(median_value)

    fill_value = "Unknown" 
    for col in categorical_features_rent:
        X_train_rent[col] = X_train_rent[col].fillna(fill_value)
        X_val_rent[col] = X_val_rent[col].fillna(fill_value)
        if col in X_test_rent.columns:
            X_test_rent[col] = X_test_rent[col].fillna(fill_value)
            
    print("第一轮缺失值填充完毕。")

    # --- 6. 检查填充结果 ---
    print("\n--- X_train_rent 填充后 info() ---")
    X_train_rent.info()

except NameError:
     print("\n错误：填充步骤因之前的错误而跳过。")

print("\n--- 基础清洗与填充阶段完成 ---")
print("下一步：特征工程 (量化, TE, OHE 等)。")


开始第一轮缺失值填充 (中位数 / 'Unknown')...
第一轮缺失值填充完毕。

--- X_train_rent 填充后 info() ---
<class 'pandas.core.frame.DataFrame'>
Index: 79119 entries, 66850 to 77652
Data columns (total 34 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   城市       79119 non-null  int64  
 1   户型       79119 non-null  object 
 2   楼层       79119 non-null  object 
 3   面积       79119 non-null  float64
 4   朝向       79119 non-null  object 
 5   交易时间     79119 non-null  object 
 6   付款方式     79119 non-null  object 
 7   租赁方式     79119 non-null  object 
 8   电梯       79119 non-null  object 
 9   用水       79119 non-null  object 
 10  用电       79119 non-null  object 
 11  燃气       79119 non-null  object 
 12  配套设施     79119 non-null  object 
 13  lon      79119 non-null  float64
 14  lat      79119 non-null  float64
 15  年份       79119 non-null  float64
 16  区县       79119 non-null  float64
 17  板块       79119 non-null  float64
 18  物业类别     79119 non-null  object 
 19  建筑年代     79

处理“户型”列

In [4]:
#定义parse_layout函数处理户型数据
try:
    parse_layout
except NameError:
    print("重新定义 parse_layout 函数...")
    def parse_layout(text_data):
        if not isinstance(text_data, str): return (np.nan, np.nan, np.nan, np.nan)
        room_match = re.search(r'(\d+)(?:室|房间)', text_data); num_rooms = int(room_match.group(1)) if room_match else 0
        living_match = re.search(r'(\d+)厅', text_data); num_living = int(living_match.group(1)) if living_match else 0
        kitchen_match = re.search(r'(\d+)厨', text_data); num_kitchen = int(kitchen_match.group(1)) if kitchen_match else 0
        bath_match = re.search(r'(\d+)卫', text_data); num_bath = int(bath_match.group(1)) if bath_match else 0
        if num_rooms == 0 and num_living == 0 and num_kitchen == 0 and num_bath == 0: return (np.nan, np.nan, np.nan, np.nan)
        return (num_rooms, num_living, num_kitchen, num_bath)

# 将函数应用到所有三个数据集 ('户型' 列)
try:
    if 'X_train_rent' not in locals(): raise NameError("X_train_rent not defined")
    
    new_cols_train_rent = X_train_rent['户型'].apply(lambda x: pd.Series(parse_layout(x), index=['Num_Rooms', 'Num_LivingRooms', 'Num_Kitchens', 'Num_Baths']))
    new_cols_val_rent = X_val_rent['户型'].apply(lambda x: pd.Series(parse_layout(x), index=['Num_Rooms', 'Num_LivingRooms', 'Num_Kitchens', 'Num_Baths']))
    new_cols_test_rent = X_test_rent['户型'].apply(lambda x: pd.Series(parse_layout(x), index=['Num_Rooms', 'Num_LivingRooms', 'Num_Kitchens', 'Num_Baths']))

    # --- 3. 合并新列 ---
    X_train_rent = pd.concat([X_train_rent, new_cols_train_rent], axis=1)
    X_val_rent = pd.concat([X_val_rent, new_cols_val_rent], axis=1)
    X_test_rent = pd.concat([X_test_rent, new_cols_test_rent], axis=1)
    print("已成功创建户型特征：'Num_Rooms', 'Num_LivingRooms', 'Num_Kitchens', 'Num_Baths'。")

    # --- 4. 填充新产生的 NaN ---
    new_layout_cols_rent = ['Num_Rooms', 'Num_LivingRooms', 'Num_Kitchens', 'Num_Baths']
    print("正在为新户型特征填充 NaN...")
    for col in new_layout_cols_rent:
        median_val = X_train_rent[col].median()
        X_train_rent[col] = X_train_rent[col].fillna(median_val)
        X_val_rent[col] = X_val_rent[col].fillna(median_val)
        if col in X_test_rent.columns: # 测试集可能没有户型列？检查一下
             X_test_rent[col] = X_test_rent[col].fillna(median_val)
        else:
             print(f"警告：列 '{col}' 不在 X_test_rent 中。")
             
    print("户型特征 NaN 填充完毕。")
    # print(X_train_rent[new_layout_cols_rent].head()) # 可选检查

except NameError as e:
     print(f"错误：变量未找到 ({e})。请确保之前的步骤已成功运行。")
except KeyError as e:
     print(f"错误：列 '{e}' 未找到。请检查租金数据的列名。")
except Exception as e:
     print(f"处理户型时发生意外错误: {e}")

重新定义 parse_layout 函数...
已成功创建户型特征：'Num_Rooms', 'Num_LivingRooms', 'Num_Kitchens', 'Num_Baths'。
正在为新户型特征填充 NaN...
户型特征 NaN 填充完毕。


处理楼层

In [5]:
# --- 处理楼层特征 ---
print("\n--- 处理楼层特征 ---")

# 定义解析总楼层数的函数
def parse_total_floors(text_data):
    if not isinstance(text_data, str): 
        return np.nan
    match = re.search(r'共(\d+)层', text_data)
    return float(match.group(1)) if match else np.nan

# 定义计算楼层比率的函数
def calculate_floor_ratio(row):
    text_data = row['楼层']
    total_floors = row['Total_Floors']
    
    if not isinstance(text_data, str):
        return np.nan
    
    # 情况1: 处理 "a/b层" 格式 (其中a,b为数字)
    match_ab = re.search(r'(\d+)/(\d+)层', text_data)
    if match_ab:
        current_floor = float(match_ab.group(1))
        building_floors = float(match_ab.group(2))
        return current_floor / building_floors
    
    # 情况2: 处理描述性楼层
    if '地下室' in text_data:
        return 0.0
    elif '底层' in text_data:
        return 1.0 / total_floors
    elif '顶层' in text_data:
        return 1.0
    elif '高楼层' in text_data:
        return 0.75
    elif '中楼层' in text_data:
        return 0.5
    elif '低楼层' in text_data:
        return 0.25
    else:
        # 既不是格式1也不是格式2的情况
        return np.nan

try:
    # 1. 提取总楼层数
    print("提取总楼层数...")
    X_train_rent['Total_Floors'] = X_train_rent['楼层'].apply(parse_total_floors)
    X_val_rent['Total_Floors'] = X_val_rent['楼层'].apply(parse_total_floors)
    X_test_rent['Total_Floors'] = X_test_rent['楼层'].apply(parse_total_floors)
    
    # 填充总楼层数的缺失值
    median_total_floors = X_train_rent['Total_Floors'].median()
    X_train_rent['Total_Floors'] = X_train_rent['Total_Floors'].fillna(median_total_floors)
    X_val_rent['Total_Floors'] = X_val_rent['Total_Floors'].fillna(median_total_floors)
    X_test_rent['Total_Floors'] = X_test_rent['Total_Floors'].fillna(median_total_floors)
    
    # 2. 计算楼层比率
    print("计算楼层比率...")
    X_train_rent['Floor_Ratio'] = X_train_rent.apply(calculate_floor_ratio, axis=1)
    X_val_rent['Floor_Ratio'] = X_val_rent.apply(calculate_floor_ratio, axis=1)
    X_test_rent['Floor_Ratio'] = X_test_rent.apply(calculate_floor_ratio, axis=1)
    
    # 填充楼层比率的缺失值
    median_floor_ratio = X_train_rent['Floor_Ratio'].median()
    X_train_rent['Floor_Ratio'] = X_train_rent['Floor_Ratio'].fillna(median_floor_ratio)
    X_val_rent['Floor_Ratio'] = X_val_rent['Floor_Ratio'].fillna(median_floor_ratio)
    X_test_rent['Floor_Ratio'] = X_test_rent['Floor_Ratio'].fillna(median_floor_ratio)
    
    # 3. 计算估计楼层
    print("计算估计楼层...")
    X_train_rent['Estimated_Floor'] = X_train_rent['Total_Floors'] * X_train_rent['Floor_Ratio']
    X_val_rent['Estimated_Floor'] = X_val_rent['Total_Floors'] * X_val_rent['Floor_Ratio']
    X_test_rent['Estimated_Floor'] = X_test_rent['Total_Floors'] * X_test_rent['Floor_Ratio']
    
    print("成功创建楼层特征: Floor_Ratio, Estimated_Floor")
    print(f"Floor_Ratio 统计: 均值={X_train_rent['Floor_Ratio'].mean():.3f}, 中位数={X_train_rent['Floor_Ratio'].median():.3f}")
    print(f"Estimated_Floor 统计: 均值={X_train_rent['Estimated_Floor'].mean():.3f}, 中位数={X_train_rent['Estimated_Floor'].median():.3f}")

except KeyError as e:
    print(f"错误：列 '{e}' 未找到。请检查数据中是否有 '所在楼层' 列。")
except Exception as e:
    print(f"处理楼层特征时发生错误: {e}")

print("\n--- 楼层特征处理完成 ---")


--- 处理楼层特征 ---
提取总楼层数...
计算楼层比率...
计算估计楼层...
成功创建楼层特征: Floor_Ratio, Estimated_Floor
Floor_Ratio 统计: 均值=0.516, 中位数=0.500
Estimated_Floor 统计: 均值=nan, 中位数=nan

--- 楼层特征处理完成 ---


处理建筑年代

In [6]:
# --- 处理建筑年代特征 ---
print("\n--- 处理建筑年代特征 ---")

# 定义解析年代函数
def parse_build_year(text_data):
    """
    解析建筑年代：
    - "2020年" -> 2020
    - "2014-2018年" -> (2014+2018)/2 = 2016
    - "Unknown" 或其他 -> np.nan
    """
    # 确保输入是字符串
    if not isinstance(text_data, str):
        return np.nan
    
    # re.findall(r'(\d{4})', text_data) 会找到所有4位数的年份
    numbers = re.findall(r'(\d{4})', text_data)
    
    if len(numbers) == 0:
        # 没找到年份 
        return np.nan
    elif len(numbers) == 1:
        # 只找到一个
        return float(numbers[0])
    else:
        # 找到多个
        # 我们取第一个和最后一个年份的平均值
        first_year = float(numbers[0])
        last_year = float(numbers[-1]) # [-1] 表示最后一个
        return (first_year + last_year) / 2

try:
    if 'X_train_rent' not in locals():
        raise NameError("X_train_rent not defined")
    
    if '建筑年代' in X_train_rent.columns:
        # 应用解析函数
        year_extracted_train = X_train_rent['建筑年代'].apply(parse_build_year)
        year_extracted_val = X_val_rent['建筑年代'].apply(parse_build_year)
        
        # 检查测试集是否有此列
        if '建筑年代' in X_test_rent.columns:
            year_extracted_test = X_test_rent['建筑年代'].apply(parse_build_year)
        else:
            # 如果测试集没有，创建一个充满 NaN 的 Series，以便后续填充
            year_extracted_test = pd.Series(np.nan, index=X_test_rent.index)
            print("警告：'建筑年代' 列不在 X_test_rent 中。")
            
        # 计算房龄 (假设当前是 2025 年)
        current_year = 2025
        X_train_rent['House_Age_Rent'] = current_year - year_extracted_train 
        X_val_rent['House_Age_Rent'] = current_year - year_extracted_val
        X_test_rent['House_Age_Rent'] = current_year - year_extracted_test
        
        print("已创建 'House_Age_Rent' 特征 (此时可能含 NaN)。")

        # 填充新产生的 NaN
        print("正在为 'House_Age_Rent' 填充 NaN...")
        col_to_fill = 'House_Age_Rent'
        
        # 计算训练集的中位数用于填充
        median_val = X_train_rent[col_to_fill].median()
        
        # 如果中位数本身是 NaN (不太可能，除非所有都是NaN)
        if np.isnan(median_val):
            median_val = 0.0  # 用 0 填充
            print("警告：House_Age_Rent 中位数为 NaN，使用 0 填充")
        
        # 填充所有数据集的 NaN 值
        X_train_rent[col_to_fill] = X_train_rent[col_to_fill].fillna(median_val)
        X_val_rent[col_to_fill] = X_val_rent[col_to_fill].fillna(median_val)
        
        # 确保测试集列存在再填充
        if col_to_fill in X_test_rent.columns:
            X_test_rent[col_to_fill] = X_test_rent[col_to_fill].fillna(median_val)
        
        print(f"已使用中位数 {median_val:.0f} 填充 '{col_to_fill}'。")
        print(f"房龄统计: 均值={X_train_rent[col_to_fill].mean():.1f}年, 中位数={X_train_rent[col_to_fill].median():.1f}年")
        
    else:
        print("跳过：'建筑年代' 列不在数据集中。")

except NameError as e:
    print(f"错误：变量未找到 ({e})。请确保之前的步骤已成功运行。")
except KeyError as e:
    print(f"错误：列 '{e}' 未找到。请检查租金数据的列名。")
except Exception as e:
    print(f"处理建筑年代时发生意外错误: {e}")

print("\n--- 建筑年代处理完成 ---")


--- 处理建筑年代特征 ---
已创建 'House_Age_Rent' 特征 (此时可能含 NaN)。
正在为 'House_Age_Rent' 填充 NaN...
已使用中位数 17 填充 'House_Age_Rent'。
房龄统计: 均值=18.4年, 中位数=17.0年

--- 建筑年代处理完成 ---


同时处理房屋总数与楼栋总数

In [7]:
# --- 处理房屋总数和楼栋总数特征 ---
print("\n--- 处理房屋总数和楼栋总数特征 ---")

# 定义提取数字的函数
def extract_number(text_data, unit):
    """
    从文本中提取数字
    - "x户" -> 提取x
    - "x栋" -> 提取x
    - 其他情况 -> np.nan
    """
    if not isinstance(text_data, str):
        return np.nan
    
    # 匹配数字后跟指定单位的模式
    pattern = r'(\d+)' + unit
    match = re.search(pattern, text_data)
    
    if match:
        return float(match.group(1))
    else:
        return np.nan

try:
    if 'X_train_rent' not in locals():
        raise NameError("X_train_rent not defined")
    
    # --- 处理房屋总数 ---
    if '房屋总数' in X_train_rent.columns:
        print("处理 '房屋总数' 列...")
        
        # 应用解析函数
        house_count_train = X_train_rent['房屋总数'].apply(lambda x: extract_number(x, '户'))
        house_count_val = X_val_rent['房屋总数'].apply(lambda x: extract_number(x, '户'))
        
        # 检查测试集是否有此列
        if '房屋总数' in X_test_rent.columns:
            house_count_test = X_test_rent['房屋总数'].apply(lambda x: extract_number(x, '户'))
        else:
            house_count_test = pd.Series(np.nan, index=X_test_rent.index)
            print("警告：'房屋总数' 列不在 X_test_rent 中。")
        
        # 创建新特征
        X_train_rent['House_Count'] = house_count_train
        X_val_rent['House_Count'] = house_count_val
        X_test_rent['House_Count'] = house_count_test
        
        # 填充缺失值
        median_house_count = X_train_rent['House_Count'].median()
        if np.isnan(median_house_count):
            median_house_count = 0.0
        
        X_train_rent['House_Count'] = X_train_rent['House_Count'].fillna(median_house_count)
        X_val_rent['House_Count'] = X_val_rent['House_Count'].fillna(median_house_count)
        X_test_rent['House_Count'] = X_test_rent['House_Count'].fillna(median_house_count)
        
        print(f"已创建 'House_Count' 特征，使用中位数 {median_house_count:.0f} 填充缺失值")
        print(f"房屋总数统计: 均值={X_train_rent['House_Count'].mean():.1f}, 中位数={X_train_rent['House_Count'].median():.1f}")
    else:
        print("跳过：'房屋总数' 列不在数据集中。")
    
    # --- 处理楼栋总数 ---
    if '楼栋总数' in X_train_rent.columns:
        print("\n处理 '楼栋总数' 列...")
        
        # 应用解析函数
        building_count_train = X_train_rent['楼栋总数'].apply(lambda x: extract_number(x, '栋'))
        building_count_val = X_val_rent['楼栋总数'].apply(lambda x: extract_number(x, '栋'))
        
        # 检查测试集是否有此列
        if '楼栋总数' in X_test_rent.columns:
            building_count_test = X_test_rent['楼栋总数'].apply(lambda x: extract_number(x, '栋'))
        else:
            building_count_test = pd.Series(np.nan, index=X_test_rent.index)
            print("警告：'楼栋总数' 列不在 X_test_rent 中。")
        
        # 创建新特征
        X_train_rent['Building_Count'] = building_count_train
        X_val_rent['Building_Count'] = building_count_val
        X_test_rent['Building_Count'] = building_count_test
        
        # 填充缺失值
        median_building_count = X_train_rent['Building_Count'].median()
        if np.isnan(median_building_count):
            median_building_count = 0.0
        
        X_train_rent['Building_Count'] = X_train_rent['Building_Count'].fillna(median_building_count)
        X_val_rent['Building_Count'] = X_val_rent['Building_Count'].fillna(median_building_count)
        X_test_rent['Building_Count'] = X_test_rent['Building_Count'].fillna(median_building_count)
        
        print(f"已创建 'Building_Count' 特征，使用中位数 {median_building_count:.0f} 填充缺失值")
        print(f"楼栋总数统计: 均值={X_train_rent['Building_Count'].mean():.1f}, 中位数={X_train_rent['Building_Count'].median():.1f}")
    else:
        print("跳过：'楼栋总数' 列不在数据集中。")

except NameError as e:
    print(f"错误：变量未找到 ({e})。请确保之前的步骤已成功运行。")
except KeyError as e:
    print(f"错误：列 '{e}' 未找到。请检查租金数据的列名。")
except Exception as e:
    print(f"处理房屋/楼栋总数时发生意外错误: {e}")

print("\n--- 房屋总数和楼栋总数处理完成 ---")


--- 处理房屋总数和楼栋总数特征 ---
处理 '房屋总数' 列...
已创建 'House_Count' 特征，使用中位数 1445 填充缺失值
房屋总数统计: 均值=2020.7, 中位数=1445.0

处理 '楼栋总数' 列...
已创建 'Building_Count' 特征，使用中位数 13 填充缺失值
楼栋总数统计: 均值=27.6, 中位数=13.0

--- 房屋总数和楼栋总数处理完成 ---


处理费用特征（燃气费、物业费）

In [8]:
# --- 处理费用特征（燃气费、物业费）---
print("\n--- 处理费用特征（燃气费、物业费）---")

# 定义提取费用数值的函数
def extract_fee_value(text_data):
    """
    从费用文本中提取数值：
    - "x元/月/㎡" 或 "x元/㎡" -> 返回x
    - "a-b元/月/㎡" 或 "a-b元/㎡" -> 返回(a+b)/2
    - 其他情况 -> np.nan
    """
    if pd.isna(text_data) or text_data == "Unknown":
        return np.nan
    
    # 转换为字符串处理
    text_str = str(text_data)
    
    # 匹配单个数字模式：数字后跟单位
    single_match = re.search(r'^(\d+\.?\d*)\s*元', text_str)
    if single_match:
        return float(single_match.group(1))
    
    # 匹配区间模式：a-b元
    range_match = re.search(r'(\d+\.?\d*)\s*-\s*(\d+\.?\d*)\s*元', text_str)
    if range_match:
        a = float(range_match.group(1))
        b = float(range_match.group(2))
        return (a + b) / 2
    
    # 如果没有匹配到任何模式，返回NaN
    return np.nan

try:
    if 'X_train_rent' not in locals():
        raise NameError("X_train_rent not defined")
    
    # 定义要处理的费用列
    fee_columns = ['燃气费', '物 业 费']
    
    for fee_col in fee_columns:
        if fee_col in X_train_rent.columns:
            print(f"\n处理 '{fee_col}' 列...")
            
            # 应用解析函数到所有数据集
            fee_train = X_train_rent[fee_col].apply(extract_fee_value)
            fee_val = X_val_rent[fee_col].apply(extract_fee_value)
            
            # 检查测试集是否有此列
            if fee_col in X_test_rent.columns:
                fee_test = X_test_rent[fee_col].apply(extract_fee_value)
            else:
                fee_test = pd.Series(np.nan, index=X_test_rent.index)
                print(f"警告：'{fee_col}' 列不在 X_test_rent 中。")
            
            # 创建新特征名称
            new_col_name = fee_col.replace('费', '') + '_Fee'
            
            # 创建新特征
            X_train_rent[new_col_name] = fee_train
            X_val_rent[new_col_name] = fee_val
            X_test_rent[new_col_name] = fee_test
            
            # 计算中位数用于填充
            median_fee = X_train_rent[new_col_name].median()
            if np.isnan(median_fee):
                median_fee = 0.0
                print(f"警告：{new_col_name} 中位数为 NaN，使用 0 填充")
            
            # 填充缺失值
            X_train_rent[new_col_name] = X_train_rent[new_col_name].fillna(median_fee)
            X_val_rent[new_col_name] = X_val_rent[new_col_name].fillna(median_fee)
            X_test_rent[new_col_name] = X_test_rent[new_col_name].fillna(median_fee)
            
            print(f"已创建 '{new_col_name}' 特征，使用中位数 {median_fee:.2f} 填充缺失值")
            print(f"{new_col_name} 统计: 均值={X_train_rent[new_col_name].mean():.2f}, 中位数={X_train_rent[new_col_name].median():.2f}")
            
            # 显示一些样本转换结果用于验证
            sample_count = min(3, len(X_train_rent))
            sample_indices = X_train_rent.head(sample_count).index
            print(f"样本转换:")
            for idx in sample_indices:
                original = X_train_rent.loc[idx, fee_col]
                converted = X_train_rent.loc[idx, new_col_name]
                print(f"  '{original}' -> {converted:.2f}")
        else:
            print(f"跳过：'{fee_col}' 列不在数据集中。")

except NameError as e:
    print(f"错误：变量未找到 ({e})。请确保之前的步骤已成功运行。")
except KeyError as e:
    print(f"错误：列 '{e}' 未找到。请检查租金数据的列名。")
except Exception as e:
    print(f"处理费用特征时发生意外错误: {e}")

print("\n--- 费用特征处理完成 ---")


--- 处理费用特征（燃气费、物业费）---

处理 '燃气费' 列...
已创建 '燃气_Fee' 特征，使用中位数 2.95 填充缺失值
燃气_Fee 统计: 均值=2.90, 中位数=2.95
样本转换:
  '3.5元/m³' -> 3.50
  '3.5元/m³' -> 3.50
  '3.45元/m³' -> 3.45

处理 '物 业 费' 列...
已创建 '物 业 _Fee' 特征，使用中位数 2.20 填充缺失值
物 业 _Fee 统计: 均值=2.69, 中位数=2.20
样本转换:
  '2.48-2.86元/月/㎡' -> 2.67
  '3.2元/月/㎡' -> 3.20
  '3元/月/㎡' -> 3.00

--- 费用特征处理完成 ---


处理绿化率与容积率,其中容积率数据类型为float，在第一步数据处理中已经完成，故只处理绿化率

In [9]:
# --- 检查容积率和绿化率的数据类型 ---
print("\n--- 检查容积率和绿化率的数据类型 ---")
if '容 积 率' in X_train_rent.columns:
    print(f"容积率的数据类型: {X_train_rent['容 积 率'].dtype}")
if '绿 化 率' in X_train_rent.columns:
    print(f"绿化率的数据类型: {X_train_rent['绿 化 率'].dtype}")

# --- 处理绿化率 ---
print("\n--- 处理绿化率 ---")

try:
    if 'X_train_rent' not in locals():
        raise NameError("X_train_rent not defined")
    
    # 处理绿化率（从百分比转换为数值）
    if '绿 化 率' in X_train_rent.columns:
        print("处理 '绿化率' 列...")
        
        # 将百分比字符串转换为数值
        def convert_green_rate(text_data):
            if pd.isna(text_data) or text_data == "Unknown":
                return np.nan
            # 移除百分号并转换为浮点数
            text_str = str(text_data).replace('%', '')
            try:
                return float(text_str)
            except ValueError:
                return np.nan
        
        # 应用到所有数据集
        X_train_rent['绿 化 率'] = X_train_rent['绿 化 率'].apply(convert_green_rate)
        X_val_rent['绿 化 率'] = X_val_rent['绿 化 率'].apply(convert_green_rate)
        if '绿 化 率' in X_test_rent.columns:
            X_test_rent['绿 化 率'] = X_test_rent['绿 化 率'].apply(convert_green_rate)
        
        # 检查并填充缺失值
        if X_train_rent['绿 化 率'].isnull().any():
            median_value = X_train_rent['绿 化 率'].median()
            X_train_rent['绿 化 率'] = X_train_rent['绿 化 率'].fillna(median_value)
            X_val_rent['绿 化 率'] = X_val_rent['绿 化 率'].fillna(median_value)
            if '绿 化 率' in X_test_rent.columns:
                X_test_rent['绿 化 率'] = X_test_rent['绿 化 率'].fillna(median_value)
            print(f"绿化率缺失值已使用中位数 {median_value:.2f} 填充")
        else:
            print("绿化率无缺失值")
            
        print(f"绿化率统计: 均值={X_train_rent['绿 化 率'].mean():.2f}, 中位数={X_train_rent['绿 化 率'].median():.2f}")
        
    else:
        print("数据集中未找到'绿化率'列")
        
except Exception as e:
    print(f"处理绿化率时发生错误: {e}")
    import traceback
    traceback.print_exc()


--- 检查容积率和绿化率的数据类型 ---
容积率的数据类型: float64
绿化率的数据类型: object

--- 处理绿化率 ---
处理 '绿化率' 列...
绿化率缺失值已使用中位数 35.00 填充
绿化率统计: 均值=43.33, 中位数=35.00


In [10]:
# --- 处理停车费用 ---
import numpy as np
print("\n--- 处理停车费用 ---")

# 定义硬编码查找表
hardcoded_parking_fees = {
    "一元钱一小时，单次24小时内最高12元一次": 12.0 * 30, 
    "小区没有停车费": 0.0,
    "无固定车位不收费": 0.0,
    "售价:30万/位；租价450元/位/月": 450.0,
    "每小时2元每个月70元": 70.0, 
    "每小时2元/位 , 每个月50元/位": 50.0, 
    "露天250元/月/位，3元/时/位；室内500元/月/位，3元/时/位": np.mean([250.0, 500.0]), 
    "露天16元/小时，室内700/月": 700.0, 
    "临保2.5元/小时月保400元/月": 400.0, 
    "临保2.5元/时/位，月保400元/月/位": 400.0, 
    "临保:4元/小时/位,月保:550元/位/月": 550.0, 
    "固定车位1400元/年，非固定车位100元/月": np.mean([1400.0 / 12.0, 100.0]), 
    "第一小时5块，后面1小时1块，一天15封顶": 15.0 * 30, 
    "地下400，地上免费": np.mean([400.0, 0.0]), 
    "地下350元/月/位 加60管理费/月": 350.0 + 60.0, 
    "地上免费  地下400元/月": np.mean([0.0, 400.0]), 
    "地上4元/小时/位": 4.0 * 8.0 * 30.0,
    "地上150元/月/位，地下2元/时/位，地下固定车位450元/月/位": np.mean([150.0, 2.0 * 8.0 * 30.0, 450.0])
}

# 定义解析函数
def parse_parking_fee_hardcoded(text_data):
    if not isinstance(text_data, str):
        return np.nan
    
    text_original = text_data.strip()
    text_lower = text_original.lower()

    # 优先级 1: 硬编码查找
    if text_original in hardcoded_parking_fees:
        return hardcoded_parking_fees[text_original]

    # 优先级 2: 明确的 0 值 (包括 '暂无')
    zero_keywords = ['免费', '没有停车费', '不收费']
    if any(keyword in text_lower for keyword in zero_keywords) or text_lower == '无' or text_lower == '暂无' or text_lower == '0':
        return 0.0
        
    # 优先级 3: 明确的未知
    unknown_keywords = ['unknown', '未知', '无法核实', '无法获知']
    if any(keyword in text_lower for keyword in unknown_keywords):
        return np.nan

    # 优先级 4: 基本数字提取与平均 (作为最后的备用)
    numbers = re.findall(r'(\d+\.?\d*)', text_original)
    numbers = [float(n) for n in numbers if n] 
    
    if len(numbers) > 0:
        return np.mean(numbers)
        
    # 优先级 5: 无法解析
    return np.nan

try:
    if 'X_train_rent' not in locals():
        raise NameError("X_train_rent not defined")

    if '停车费用' in X_train_rent.columns:
        # 应用解析函数
        X_train_rent['Parking_Fee_Rent'] = X_train_rent['停车费用'].apply(parse_parking_fee_hardcoded)
        X_val_rent['Parking_Fee_Rent'] = X_val_rent['停车费用'].apply(parse_parking_fee_hardcoded)
        
        # 检查测试集
        if '停车费用' in X_test_rent.columns:
            X_test_rent['Parking_Fee_Rent'] = X_test_rent['停车费用'].apply(parse_parking_fee_hardcoded)
        else:
            X_test_rent['Parking_Fee_Rent'] = np.nan
            print("警告：'停车费用' 列不在 X_test_rent 中。")

        print("已创建 'Parking_Fee_Rent' 特征 (此时可能含 NaN)。")

        # 填充缺失值
        print("正在为 'Parking_Fee_Rent' 填充 NaN...")
        col_to_fill = 'Parking_Fee_Rent'
        if X_train_rent[col_to_fill].isnull().any():
            median_val = X_train_rent[col_to_fill].median()
            fill_val = median_val if not np.isnan(median_val) else 300.0 # 停车费默认填充 300
            
            X_train_rent[col_to_fill] = X_train_rent[col_to_fill].fillna(fill_val)
            X_val_rent[col_to_fill] = X_val_rent[col_to_fill].fillna(fill_val)
            if col_to_fill in X_test_rent.columns:
                X_test_rent[col_to_fill] = X_test_rent[col_to_fill].fillna(fill_val)
            print(f"已使用中位数 {fill_val:.2f} 填充 '{col_to_fill}'。")
        else:
            print(f"'{col_to_fill}' 中无 NaN 需要填充。")
            
        print(f"停车费用统计: 均值={X_train_rent[col_to_fill].mean():.2f}, 中位数={X_train_rent[col_to_fill].median():.2f}")

    else:
        print("跳过：'停车费用' 列不在数据集中。")

except NameError as e:
    print(f"错误：变量未找到 ({e})。请确保之前的步骤已成功运行。")
except KeyError as e:
    print(f"错误：列 '{e}' 未找到。请检查租金数据的列名。")
except Exception as e:
    print(f"处理停车费用时发生意外错误: {e}")

print("\n--- 停车费用处理完成 ---")


--- 处理停车费用 ---
已创建 'Parking_Fee_Rent' 特征 (此时可能含 NaN)。
正在为 'Parking_Fee_Rent' 填充 NaN...
已使用中位数 300.00 填充 'Parking_Fee_Rent'。
停车费用统计: 均值=318.80, 中位数=300.00

--- 停车费用处理完成 ---


In [11]:
print("对目标变量 (Rent) 进行 Log Transformation")

try:
    # y_train_rent 和 y_val_rent 应该包含原始租金值
    y_train_rent_log = np.log1p(y_train_rent)
    y_val_rent_log = np.log1p(y_val_rent)
    
    print("Rent 目标变量已成功进行 log1p 转换。")
    
    # 计算全局均值 (用于平滑)
    global_mean_rent_log = y_train_rent_log.mean()
    print(f"全局平均对数租金 (Global Mean for Smoothing): {global_mean_rent_log:.4f}")

except NameError as e:
    print(f"错误：变量未找到 ({e})。请确保分割步骤已成功运行。")
except Exception as e:
    print(f"对数转换时发生意外错误: {e}")

对目标变量 (Rent) 进行 Log Transformation
Rent 目标变量已成功进行 log1p 转换。
全局平均对数租金 (Global Mean for Smoothing): 12.9589


In [12]:
# --- 执行目标编码 (城市/区县/板块) ---
print("\n--- 执行目标编码 (城市/区县/板块) ---")

try:
    if 'X_train_rent' not in locals():
        raise NameError("X_train_rent not defined")
    
    # 确保目标变量已定义
    if 'y_train_rent' not in locals():
        raise NameError("y_train_rent not defined")
    
    # 1. 准备数据
    target_col_rent_name = 'Rent'
    global_mean_rent = y_train_rent.mean()
    
    # 对租金取对数（如果数据有偏斜）
    y_train_rent_log = np.log1p(y_train_rent)
    global_mean_rent_log = y_train_rent_log.mean()
    
    train_data_for_encoding_rent = pd.concat([X_train_rent, y_train_rent_log], axis=1)
    
    # 2. 目标编码 '城市'
    m_city = 20
    city_stats_rent = train_data_for_encoding_rent.groupby('城市')[target_col_rent_name].agg(['mean', 'count'])
    city_stats_rent['TE_City_Rent'] = (city_stats_rent['count'] * city_stats_rent['mean'] + m_city * global_mean_rent_log) / (city_stats_rent['count'] + m_city)
    city_encoding_map_rent = city_stats_rent['TE_City_Rent'].to_dict()
    
    # 应用 '城市' 编码
    X_train_rent['TE_City_Rent'] = X_train_rent['城市'].map(city_encoding_map_rent)
    X_val_rent['TE_City_Rent'] = X_val_rent['城市'].map(city_encoding_map_rent).fillna(global_mean_rent_log) 
    X_test_rent['TE_City_Rent'] = X_test_rent['城市'].map(city_encoding_map_rent).fillna(global_mean_rent_log) 
    print("✅ 'TE_City_Rent' 创建完成。")
    
    # 3. 目标编码 '区县'
    m_district = 20
    district_stats_rent = train_data_for_encoding_rent.groupby('区县')[target_col_rent_name].agg(['mean', 'count'])
    district_stats_rent['TE_District_Rent'] = (district_stats_rent['count'] * district_stats_rent['mean'] + m_district * global_mean_rent_log) / (district_stats_rent['count'] + m_district)
    district_encoding_map_rent = district_stats_rent['TE_District_Rent'].to_dict()
    
    # 应用 '区县' 编码
    X_train_rent['TE_District_Rent'] = X_train_rent['区县'].map(district_encoding_map_rent)
    X_val_rent['TE_District_Rent'] = X_val_rent['区县'].map(district_encoding_map_rent).fillna(global_mean_rent_log) 
    X_test_rent['TE_District_Rent'] = X_test_rent['区县'].map(district_encoding_map_rent).fillna(global_mean_rent_log) 
    print("✅ 'TE_District_Rent' 创建完成。")
    
    # 4. 目标编码 '板块'
    m_block = 20
    block_stats_rent = train_data_for_encoding_rent.groupby(['区县', '板块'])[target_col_rent_name].agg(['mean', 'count'])
    district_original_mean_map_rent = district_stats_rent['mean'].to_dict()
    
    # 分层平滑函数
    def stratified_smooth_rent(row, district_means, global_m, global_mean):
        block_mean = row['mean']
        block_count = row['count']
        district_id = row.name[0] 
        district_mean = district_means.get(district_id, global_mean) 
        smoothed_mean = (block_count * block_mean + global_m * district_mean) / (block_count + global_m)
        return smoothed_mean
    
    block_stats_rent['TE_Block_Stratified_Rent'] = block_stats_rent.apply(
        stratified_smooth_rent, axis=1, 
        district_means=district_original_mean_map_rent, 
        global_m=m_block, 
        global_mean=global_mean_rent_log
    )
    block_encoding_map_rent = block_stats_rent['TE_Block_Stratified_Rent'].to_dict()
    
    # 应用 '板块' 编码
    def apply_block_encoding_rent(row, block_map, district_map_smoothed, global_mean_val):
        key = (row['区县'], row['板块'])
        block_value = block_map.get(key)
        if block_value is not None:
            return block_value
        else:
            district_value = district_map_smoothed.get(row['区县'])
            if district_value is not None:
                return district_value
            else:
                return global_mean_val
    
    X_train_rent['TE_Block_Stratified_Rent'] = X_train_rent.apply(
        apply_block_encoding_rent, axis=1, 
        block_map=block_encoding_map_rent, 
        district_map_smoothed=district_encoding_map_rent, 
        global_mean_val=global_mean_rent_log
    )
    X_val_rent['TE_Block_Stratified_Rent'] = X_val_rent.apply(
        apply_block_encoding_rent, axis=1, 
        block_map=block_encoding_map_rent, 
        district_map_smoothed=district_encoding_map_rent, 
        global_mean_val=global_mean_rent_log
    )
    X_test_rent['TE_Block_Stratified_Rent'] = X_test_rent.apply(
        apply_block_encoding_rent, axis=1, 
        block_map=block_encoding_map_rent, 
        district_map_smoothed=district_encoding_map_rent, 
        global_mean_val=global_mean_rent_log
    )
    print("✅ 'TE_Block_Stratified_Rent' 创建完成。")
    
    # 5. 检查结果
    print(f"\n目标编码完成。检查 NaN 数量:")
    print(f"X_train: {X_train_rent['TE_Block_Stratified_Rent'].isnull().sum()}")
    print(f"X_val: {X_val_rent['TE_Block_Stratified_Rent'].isnull().sum()}")
    print(f"X_test: {X_test_rent['TE_Block_Stratified_Rent'].isnull().sum()}")

except NameError as e:
    print(f"错误：变量未找到 ({e})。请确保之前的步骤已成功运行。")
except KeyError as e:
    print(f"错误：列 '{e}' 未找到。请检查数据中是否有 '城市'、'区县'、'板块' 列。")
except Exception as e:
    print(f"执行目标编码时发生错误: {e}")

print("\n--- 目标编码完成 ---")


--- 执行目标编码 (城市/区县/板块) ---
✅ 'TE_City_Rent' 创建完成。
✅ 'TE_District_Rent' 创建完成。
✅ 'TE_Block_Stratified_Rent' 创建完成。

目标编码完成。检查 NaN 数量:
X_train: 0
X_val: 0
X_test: 0

--- 目标编码完成 ---


In [13]:
# --- 处理配套设施特征 ---
print("\n--- 处理配套设施特征 ---")

try:
    if 'X_train_rent' not in locals():
        raise NameError("X_train_rent not defined")
    
    # 检查是否存在配套设施列
    if '配套设施' not in X_train_rent.columns:
        print("跳过：'配套设施' 列不在数据集中。")
    else:
        # 定义所有可能的设施
        facility_list = ['洗衣机', '空调', '衣柜', '电视', '热水器', '床', '宽带', '冰箱', '暖气', '天然气']
        
        # 处理函数（向量化方法）
        def extract_facilities_vectorized(df, facility_list):
            """
            使用向量化方法从配套设施文本中提取设备存在情况
            """
            # 创建设施列
            for facility in facility_list:
                col_name = f'Has_{facility}'
                # 检查是否包含设施，注意处理缺失值
                df[col_name] = df['配套设施'].fillna('').str.contains(facility, na=False).astype(int)
            return df
        
        print("正在提取配套设施特征...")
        
        # 应用处理函数到所有数据集
        X_train_rent = extract_facilities_vectorized(X_train_rent, facility_list)
        X_val_rent = extract_facilities_vectorized(X_val_rent, facility_list)
        if '配套设施' in X_test_rent.columns:
            X_test_rent = extract_facilities_vectorized(X_test_rent, facility_list)
        else:
            # 如果测试集没有配套设施列，创建全0的特征
            for facility in facility_list:
                col_name = f'Has_{facility}'
                X_test_rent[col_name] = 0
            print("警告：'配套设施' 列不在 X_test_rent 中，已创建全0的特征。")
        
        # 创建设施总数衍生特征
        facility_cols = [f'Has_{facility}' for facility in facility_list]
        X_train_rent['Facility_Count'] = X_train_rent[facility_cols].sum(axis=1)
        X_val_rent['Facility_Count'] = X_val_rent[facility_cols].sum(axis=1)
        X_test_rent['Facility_Count'] = X_test_rent[facility_cols].sum(axis=1)
        
        # ... 其余统计和显示代码保持不变 ...
        
except Exception as e:
    print(f"处理配套设施时发生错误: {e}")
    import traceback
    traceback.print_exc()

print("\n--- 配套设施处理完成 ---")


--- 处理配套设施特征 ---
正在提取配套设施特征...

--- 配套设施处理完成 ---


In [16]:
try:
    final_numeric_features = X_train_rent.select_dtypes(include=['int64', 'float64']).columns.tolist()
    
    # 🆗 新增：确保配套设施特征被包含（不删除原有代码）
    # 检查并添加新创建的配套设施特征
    facility_features = [f'Has_{facility}' for facility in ['洗衣机', '空调', '衣柜', '电视', '热水器', '床', '宽带', '冰箱', '暖气', '天然气']]
    facility_features.append('Facility_Count')
    
    added_features = []
    for feat in facility_features:
        if feat in X_train_rent.columns and feat not in final_numeric_features:
            final_numeric_features.append(feat)
            added_features.append(feat)
    
    if added_features:
        print(f"✅ 自动添加配套设施特征: {added_features}")
    
except NameError:
    print("错误：X_train_rent 未定义。请确保 X_train_rent 已被加载。")
    final_numeric_features = []

object_cols_to_drop = ['物业公司', '开发商','房屋总数','停车费','燃气费','户型','配套设施','物业类别','建筑年代','楼栋总数','建筑结构','产权描述','停车费用'] 

try:
    all_object_cols = X_train_rent.select_dtypes(include=['object']).columns.tolist()
    special_handling_cols = ['楼层', '物 业 费', '交易时间']
    
    # 筛选出需要OHE的列：即所有object列中，排除掉那些您准备特殊处理的列
    categorical_for_ohe = [col for col in all_object_cols if col not in special_handling_cols]
except NameError:
    # 如果 X_train_rent 未定义，也初始化为空列表
    categorical_for_ohe = []
    
print("--- 补充变量定义完毕 ---")
print(f"已定义 final_numeric_features (初始): {len(final_numeric_features)}个特征")
print(f"已定义 object_cols_to_drop (初始): {object_cols_to_drop}")
print(f"已定义 categorical_for_ohe (初始): {len(categorical_for_ohe)}个特征")

# --- 优化高基数特征 ---
print("\n--- 优化高基数特征 ---")

from sklearn.preprocessing import StandardScaler

try:
    # --- 1. 处理高基数特征 ---
    # 定义高基数阈值
    HIGH_CARDINALITY_THRESHOLD = 20
    
    # 识别高基数特征
    high_cardinality_cols = []
    for col in categorical_for_ohe:
        if X_train_rent[col].nunique() > HIGH_CARDINALITY_THRESHOLD:
            high_cardinality_cols.append(col)
    
    print(f"高基数特征: {high_cardinality_cols}")
    
    # 处理策略：
    # - 楼层: 已从楼层特征中提取了数值特征，可以删除原始列
    # - 物业费: 已提取数值特征，可以删除原始列  
    # - 交易时间: 可以转换为数值特征（提取年份月份）
    
    # --- 2. 特殊处理交易时间 ---
    if '交易时间' in X_train_rent.columns:
        print("处理 '交易时间' 特征...")
        
        def extract_time_features(df):
            df = df.copy()
            # 提取年份和月份
            df['交易时间'] = pd.to_datetime(df['交易时间'], errors='coerce')
            df['交易年份'] = df['交易时间'].dt.year
            df['交易月份'] = df['交易时间'].dt.month
            # 计算距今的时间（月数）
            current_time = pd.Timestamp('2025-01-01')  # 假设当前时间
            df['交易距今月数'] = ((current_time - df['交易时间']).dt.days / 30).fillna(0)
            return df.drop(columns=['交易时间'])
        
        X_train_rent = extract_time_features(X_train_rent)
        X_val_rent = extract_time_features(X_val_rent)
        X_test_rent = extract_time_features(X_test_rent)
        
        # 添加到数值特征列表
        time_features = ['交易年份', '交易月份', '交易距今月数']
        for feat in time_features:
            if feat not in final_numeric_features:
                final_numeric_features.append(feat)
    
    # --- 3. 更新要删除的object列 ---
    # 删除已被处理的高基数特征
    object_cols_to_drop.extend(['楼层', '物 业 费', '交易时间'])
    
    # --- 4. 重新识别OHE列 ---
    current_object_cols = X_train_rent.select_dtypes(include=['object']).columns.tolist()
    categorical_for_ohe_optimized = [col for col in current_object_cols if col not in object_cols_to_drop]
    
    print(f"优化后OHE列 ({len(categorical_for_ohe_optimized)}个): {categorical_for_ohe_optimized}")
    
    # 检查优化后OHE特征的基数
    print("\n优化后OHE特征基数:")
    total_ohe_after_optimization = 0
    for col in categorical_for_ohe_optimized:
        unique_count = X_train_rent[col].nunique()
        ohe_features = max(0, unique_count - 1)
        total_ohe_after_optimization += ohe_features
        print(f"  {col}: {unique_count} 唯一值 -> {ohe_features} OHE特征")
    
    print(f"\n优化前OHE特征总数: ~2328")
    print(f"优化后OHE特征总数: {total_ohe_after_optimization}")
    print(f"预计总特征数: {len(final_numeric_features) + total_ohe_after_optimization}")
    
    # --- 5. 删除全零列 ---
    if 'Num_Kitchens' in final_numeric_features:
        final_numeric_features.remove('Num_Kitchens')
        print(f"删除全零列: Num_Kitchens")
    
    # --- 6. 重新执行数据准备流程 ---
    all_final_features_optimized = final_numeric_features + categorical_for_ohe_optimized
    
    # 创建清理后的数据集
    X_train_final_opt = X_train_rent[all_final_features_optimized].copy()
    X_val_final_opt = X_val_rent[all_final_features_optimized].copy()
    X_test_final_opt = X_test_rent[all_final_features_optimized].copy()
    
    print(f"优化后清理维度: 训练集{X_train_final_opt.shape}")
    
    # --- 7. One-Hot编码 ---
    print("执行优化One-Hot编码...")
    if categorical_for_ohe_optimized:
        X_train_encoded_opt = pd.get_dummies(X_train_final_opt, columns=categorical_for_ohe_optimized, 
                                           drop_first=True, dummy_na=False)
        X_val_encoded_opt = pd.get_dummies(X_val_final_opt, columns=categorical_for_ohe_optimized, 
                                         drop_first=True, dummy_na=False)
        X_test_encoded_opt = pd.get_dummies(X_test_final_opt, columns=categorical_for_ohe_optimized, 
                                          drop_first=True, dummy_na=False)
        
        # 对齐所有数据集的列
        train_columns_opt = X_train_encoded_opt.columns
        X_val_encoded_opt = X_val_encoded_opt.reindex(columns=train_columns_opt, fill_value=0)
        X_test_encoded_opt = X_test_encoded_opt.reindex(columns=train_columns_opt, fill_value=0)
    else:
        X_train_encoded_opt = X_train_final_opt.copy()
        X_val_encoded_opt = X_val_final_opt.copy()
        X_test_encoded_opt = X_test_final_opt.copy()
        train_columns_opt = X_train_encoded_opt.columns
    
    print(f"优化后编码维度: {X_train_encoded_opt.shape}")
    
    # --- 8. 缩尾处理 ---
    print("执行缩尾处理...")
    X_train_winsorized_opt = X_train_encoded_opt.copy()
    X_val_winsorized_opt = X_val_encoded_opt.copy()
    X_test_winsorized_opt = X_test_encoded_opt.copy()
    
    for col in final_numeric_features:
        if col in X_train_encoded_opt.columns:
            lower = X_train_encoded_opt[col].quantile(0.01)
            upper = X_train_encoded_opt[col].quantile(0.99)
            X_train_winsorized_opt[col] = X_train_encoded_opt[col].clip(lower, upper)
            X_val_winsorized_opt[col] = X_val_encoded_opt[col].clip(lower, upper)
            X_test_winsorized_opt[col] = X_test_encoded_opt[col].clip(lower, upper)
    
    # --- 9. 特征标准化 ---
    print("执行特征标准化...")
    scaler_opt = StandardScaler()
    X_train_scaled_opt = scaler_opt.fit_transform(X_train_winsorized_opt)
    X_val_scaled_opt = scaler_opt.transform(X_val_winsorized_opt)
    X_test_scaled_opt = scaler_opt.transform(X_test_winsorized_opt)
    
    # 转换回DataFrame
    X_train_prepared_opt = pd.DataFrame(X_train_scaled_opt, index=X_train_winsorized_opt.index, columns=train_columns_opt)
    X_val_prepared_opt = pd.DataFrame(X_val_scaled_opt, index=X_val_winsorized_opt.index, columns=train_columns_opt)
    X_test_prepared_opt = pd.DataFrame(X_test_scaled_opt, index=X_test_winsorized_opt.index, columns=train_columns_opt)
    
    print(f"最终特征维度: {X_train_prepared_opt.shape}")

except Exception as e:
    print(f"优化处理过程中发生错误: {e}")
    import traceback
    traceback.print_exc()

print("\n--- 特征优化完成 ---")

--- 补充变量定义完毕 ---
已定义 final_numeric_features (初始): 40个特征
已定义 object_cols_to_drop (初始): ['物业公司', '开发商', '房屋总数', '停车费', '燃气费', '户型', '配套设施', '物业类别', '建筑年代', '楼栋总数', '建筑结构', '产权描述', '停车费用']
已定义 categorical_for_ohe (初始): 19个特征

--- 优化高基数特征 ---
高基数特征: ['户型', '朝向', '配套设施', '物业类别', '建筑年代', '开发商', '房屋总数', '楼栋总数', '物业公司', '产权描述', '燃气费', '停车费用']
优化后OHE列 (7个): ['朝向', '付款方式', '租赁方式', '电梯', '用水', '用电', '燃气']

优化后OHE特征基数:
  朝向: 88 唯一值 -> 87 OHE特征
  付款方式: 8 唯一值 -> 7 OHE特征
  租赁方式: 2 唯一值 -> 1 OHE特征
  电梯: 3 唯一值 -> 2 OHE特征
  用水: 3 唯一值 -> 2 OHE特征
  用电: 3 唯一值 -> 2 OHE特征
  燃气: 3 唯一值 -> 2 OHE特征

优化前OHE特征总数: ~2328
优化后OHE特征总数: 103
预计总特征数: 143
删除全零列: Num_Kitchens
优化后清理维度: 训练集(79119, 46)
执行优化One-Hot编码...
优化后编码维度: (79119, 142)
执行缩尾处理...
执行特征标准化...


d:\anaconda\Lib\site-packages\sklearn\utils\extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
d:\anaconda\Lib\site-packages\sklearn\utils\extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
d:\anaconda\Lib\site-packages\sklearn\utils\extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


最终特征维度: (79119, 142)

--- 特征优化完成 ---


开始建立模型

In [ ]:
# --- 租金预测建模与评估 ---
print("\n--- 租金预测建模与评估 ---")

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("1. 数据质量检查和清理...")

# 确保使用正确的数据
if 'X_train_prepared_opt' in locals():
    X_train_final = X_train_prepared_opt.copy()
    X_val_final = X_val_prepared_opt.copy()
    X_test_final = X_test_prepared_opt.copy()
elif 'X_train_prepared' in locals():
    X_train_final = X_train_prepared.copy()
    X_val_final = X_val_prepared.copy()
    X_test_final = X_test_prepared.copy()
else:
    print("错误：未找到准备好的数据")
    exit()

print(f"原始数据形状 - 训练集: {X_train_final.shape}, 验证集: {X_val_final.shape}, 测试集: {X_test_final.shape}")

# 目标变量对数变换
y_train_log = np.log1p(y_train_rent)
y_val_log = np.log1p(y_val_rent)

# 简化的数据清理函数 - 保持列数不变
def simple_data_cleaning(X_train, X_val, X_test):
    """简化但可靠的数据清理"""
    print("执行简化数据清理...")
    
    # 创建副本
    X_train_clean = X_train.copy()
    X_val_clean = X_val.copy()
    X_test_clean = X_test.copy()
    
    # 1. 首先处理无穷大值
    for df in [X_train_clean, X_val_clean, X_test_clean]:
        for col in df.columns:
            if df[col].dtype in [np.float64, np.int64]:
                # 替换无穷大为NaN
                df[col] = df[col].replace([np.inf, -np.inf], np.nan)
    
    # 2. 直接使用列中位数填充NaN，保持列数不变
    for col in X_train_clean.columns:
        if X_train_clean[col].dtype in [np.float64, np.int64]:
            # 计算训练集的中位数
            median_val = X_train_clean[col].median()
            
            # 如果中位数是NaN，使用0
            if pd.isna(median_val):
                median_val = 0
                
            # 填充所有数据集的NaN
            X_train_clean[col] = X_train_clean[col].fillna(median_val)
            X_val_clean[col] = X_val_clean[col].fillna(median_val)
            X_test_clean[col] = X_test_clean[col].fillna(median_val)
    
    # 3. 验证清理结果
    train_nan_after = X_train_clean.isnull().sum().sum()
    val_nan_after = X_val_clean.isnull().sum().sum()
    test_nan_after = X_test_clean.isnull().sum().sum()
    
    print(f"清理后 - 训练集NaN: {train_nan_after}")
    print(f"清理后 - 验证集NaN: {val_nan_after}")
    print(f"清理后 - 测试集NaN: {test_nan_after}")
    print(f"清理后形状 - 训练集: {X_train_clean.shape}, 验证集: {X_val_clean.shape}, 测试集: {X_test_clean.shape}")
    
    return X_train_clean, X_val_clean, X_test_clean

# 执行简化的数据清理
X_train_clean, X_val_clean, X_test_clean = simple_data_cleaning(X_train_final, X_val_final, X_test_final)

# 转换为numpy数组用于建模
X_train_array = X_train_clean.values
X_val_array = X_val_clean.values
X_test_array = X_test_clean.values

print(f"最终数据形状 - 训练集: {X_train_array.shape}, 验证集: {X_val_array.shape}, 测试集: {X_test_array.shape}")

# 定义评估函数
def evaluate_model(model, X_train, y_train, X_val, y_val, model_name):
    """评估模型性能"""
    
    try:
        # 训练集预测
        y_train_pred = model.predict(X_train)
        train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
        train_r2 = r2_score(y_train, y_train_pred)
        
        # 验证集预测
        y_val_pred = model.predict(X_val)
        val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
        val_r2 = r2_score(y_val, y_val_pred)
        val_mae = mean_absolute_error(y_val, y_val_pred)
        
        results = {
            'model_name': model_name,
            'train_rmse': train_rmse,
            'val_rmse': val_rmse,
            'train_r2': train_r2,
            'val_r2': val_r2,
            'val_mae': val_mae
        }
        
        return results, y_val_pred
    except Exception as e:
        print(f"  评估模型时出错: {e}")
        return None, None

# 🔥 新增：生成测试集预测文件的函数
def generate_test_predictions(model, X_test, model_name, test_ids):
    """生成测试集预测并保存为CSV文件"""
    try:
        # 生成预测（对数空间）
        test_predictions_log = model.predict(X_test)
        # 转换回原始空间
        test_predictions = np.expm1(test_predictions_log)
        
        # 确保预测值是合理的
        if np.any(np.isnan(test_predictions)) or np.any(np.isinf(test_predictions)):
            print(f"  警告：{model_name}预测结果包含NaN或无穷大值，进行清理")
            test_predictions = np.nan_to_num(test_predictions, nan=np.nanmedian(test_predictions))
            test_predictions = np.clip(test_predictions, test_predictions.min(), test_predictions.max())
        
        # 创建提交文件
        submission_df = pd.DataFrame({
            'ID': test_ids,
            'Rent_Prediction': test_predictions
        })
        
        # 生成文件名（去掉空格）
        filename = f"rent_predictions_{model_name.replace(' ', '_').lower()}.csv"
        submission_df.to_csv(filename, index=False)
        
        print(f"  ✓ {model_name}预测结果已保存到: {filename}")
        print(f"  ✓ 预测样本数量: {len(test_predictions)}")
        print(f"  ✓ 预测租金范围: {test_predictions.min():.2f} - {test_predictions.max():.2f}")
        
        return test_predictions
        
    except Exception as e:
        print(f"  {model_name}测试集预测出错: {e}")
        return None

print("\n2. 模型训练和优化...")

models_results = []
trained_models = {}  # 🔥 新增：存储所有训练好的模型

print("\n--- 线性回归 ---")
try:
    linear_model = LinearRegression()
    linear_model.fit(X_train_array, y_train_log)
    linear_results, linear_pred = evaluate_model(linear_model, X_train_array, y_train_log, X_val_array, y_val_log, "Linear Regression")
    if linear_results:
        models_results.append(linear_results)
        trained_models['linear'] = linear_model  # 🔥 存储模型
        print(f"  RMSE: {linear_results['val_rmse']:.4f}, R²: {linear_results['val_r2']:.4f}")
        
        # 🔥 新增：立即生成线性回归的测试集预测
        print("  生成线性回归测试集预测...")
        generate_test_predictions(linear_model, X_test_array, "linear_regression", test_ids_rent)
    else:
        print("  线性回归训练失败")
except Exception as e:
    print(f"  线性回归训练出错: {e}")

print("\n--- Lasso回归 ---")
try:
    lasso_params = {'alpha': [0.001, 0.01, 0.1, 1, 10]}
    lasso_grid = GridSearchCV(Lasso(random_state=111, max_iter=1000), lasso_params, 
                              cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
    lasso_grid.fit(X_train_array, y_train_log)
    lasso_model = lasso_grid.best_estimator_
    lasso_results, lasso_pred = evaluate_model(lasso_model, X_train_array, y_train_log, X_val_array, y_val_log, "Lasso")
    if lasso_results:
        models_results.append(lasso_results)
        trained_models['lasso'] = lasso_model  # 🔥 存储模型
        print(f"  最佳alpha: {lasso_grid.best_params_['alpha']}")
        print(f"  RMSE: {lasso_results['val_rmse']:.4f}, R²: {lasso_results['val_r2']:.4f}")
        
        # 🔥 新增：立即生成Lasso回归的测试集预测
        print("  生成Lasso回归测试集预测...")
        generate_test_predictions(lasso_model, X_test_array, "lasso", test_ids_rent)
    else:
        print("  Lasso训练失败")
except Exception as e:
    print(f"  Lasso训练出错: {e}")

print("\n--- 岭回归 ---")
try:
    ridge_params = {'alpha': [0.001, 0.01, 0.1, 1, 10]}
    ridge_grid = GridSearchCV(Ridge(random_state=111), ridge_params, 
                             cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
    ridge_grid.fit(X_train_array, y_train_log)
    ridge_model = ridge_grid.best_estimator_
    ridge_results, ridge_pred = evaluate_model(ridge_model, X_train_array, y_train_log, X_val_array, y_val_log, "Ridge")
    if ridge_results:
        models_results.append(ridge_results)
        trained_models['ridge'] = ridge_model  # 🔥 存储模型
        print(f"  最佳alpha: {ridge_grid.best_params_['alpha']}")
        print(f"  RMSE: {ridge_results['val_rmse']:.4f}, R²: {ridge_results['val_r2']:.4f}")
        
        # 🔥 新增：立即生成岭回归的测试集预测
        print("  生成岭回归测试集预测...")
        generate_test_predictions(ridge_model, X_test_array, "ridge", test_ids_rent)
    else:
        print("  岭回归训练失败")
except Exception as e:
    print(f"  岭回归训练出错: {e}")

print("\n--- 弹性网络 ---")
try:
    elastic_params = {
        'alpha': [0.001, 0.01, 0.1],
        'l1_ratio': [0.1, 0.5, 0.9]
    }
    elastic_grid = GridSearchCV(ElasticNet(random_state=111, max_iter=1000), elastic_params,
                               cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
    elastic_grid.fit(X_train_array, y_train_log)
    elastic_model = elastic_grid.best_estimator_
    elastic_results, elastic_pred = evaluate_model(elastic_model, X_train_array, y_train_log, X_val_array, y_val_log, "ElasticNet")
    if elastic_results:
        models_results.append(elastic_results)
        trained_models['elasticnet'] = elastic_model  # 🔥 存储模型
        print(f"  最佳参数: alpha={elastic_grid.best_params_['alpha']}, l1_ratio={elastic_grid.best_params_['l1_ratio']}")
        print(f"  RMSE: {elastic_results['val_rmse']:.4f}, R²: {elastic_results['val_r2']:.4f}")
        
        # 🔥 新增：立即生成弹性网络的测试集预测
        print("  生成弹性网络测试集预测...")
        generate_test_predictions(elastic_model, X_test_array, "elasticnet", test_ids_rent)
    else:
        print("  弹性网络训练失败")
except Exception as e:
    print(f"  弹性网络训练出错: {e}")


# 检查是否有成功的模型
if not models_results:
    print("所有模型训练都失败了！")
    exit()

print("\n3. 模型比较和分析...")

# 创建结果比较表格
results_df = pd.DataFrame(models_results)
results_df = results_df.sort_values('val_rmse')

print("\n" + "="*80)
print("模型性能比较")
print("="*80)
print(results_df.round(4))

# 找到最佳模型
best_model_idx = results_df['val_rmse'].idxmin()
best_model_name = results_df.loc[best_model_idx, 'model_name']

# 获取最佳模型
model_mapping = {
    'Linear Regression': trained_models.get('linear'),
    'Lasso': trained_models.get('lasso'),
    'Ridge': trained_models.get('ridge'),
    'ElasticNet': trained_models.get('elasticnet'),
}

best_model = model_mapping.get(best_model_name)

if best_model is None:
    print(f"警告：无法找到最佳模型 {best_model_name}，使用第一个可用模型")
    best_model_name = results_df.iloc[0]['model_name']
    best_model = model_mapping.get(best_model_name)

print(f"\n🏆 最佳模型: {best_model_name}")
print(f"   验证集RMSE: {results_df.loc[best_model_idx, 'val_rmse']:.4f}")
print(f"   验证集R²: {results_df.loc[best_model_idx, 'val_r2']:.4f}")
print(f"   验证集MAE: {results_df.loc[best_model_idx, 'val_mae']:.4f}")

print("\n4. 生成最佳模型的测试集预测结果...")

# 使用最佳模型进行测试集预测（作为主要提交文件）
try:
    test_predictions_log = best_model.predict(X_test_array)
    test_predictions = np.expm1(test_predictions_log)
    
    # 确保预测值是合理的
    if np.any(np.isnan(test_predictions)) or np.any(np.isinf(test_predictions)):
        print("警告：预测结果包含NaN或无穷大值，进行清理")
        test_predictions = np.nan_to_num(test_predictions, nan=np.nanmedian(test_predictions))
        test_predictions = np.clip(test_predictions, test_predictions.min(), test_predictions.max())
    
    # 创建提交文件
    submission_df = pd.DataFrame({
        'ID': test_ids_rent,
        'Rent_Prediction': test_predictions
    })
    
    # 保存预测结果（最佳模型）
    submission_df.to_csv('rent_final_predictions_best.csv', index=False)
    
    print(f"✓ 最佳模型预测结果已保存到: rent_final_predictions_best.csv")
    print(f"✓ 预测样本数量: {len(test_predictions)}")
    print(f"✓ 预测租金范围: {test_predictions.min():.2f} - {test_predictions.max():.2f}")
    
except Exception as e:
    print(f"测试集预测出错: {e}")

print("\n" + "="*80)
print("🎯 租金预测建模完成!")
print("="*80)
print(f"📊 使用的特征数量: {X_train_array.shape[1]}")
print(f"🏆 最佳模型: {best_model_name}")
print(f"📈 验证集性能:")
print(f"   - RMSE: {results_df.loc[best_model_idx, 'val_rmse']:.4f}")
print(f"   - R²: {results_df.loc[best_model_idx, 'val_r2']:.4f}")
print(f"   - MAE: {results_df.loc[best_model_idx, 'val_mae']:.4f}")
print(f"💾 输出文件:")
print(f"   - 最佳模型预测: rent_final_predictions_best.csv")
print(f"   - 线性回归预测: rent_predictions_linear_regression.csv")
print(f"   - Lasso回归预测: rent_predictions_lasso.csv")
print(f"   - 岭回归预测: rent_predictions_ridge.csv")
print(f"   - 弹性网络预测: rent_predictions_elasticnet.csv")
print("="*80)


--- 租金预测建模与评估 ---
1. 数据质量检查和清理...
原始数据形状 - 训练集: (79119, 142), 验证集: (19780, 142), 测试集: (9773, 142)
执行简化数据清理...
清理后 - 训练集NaN: 0
清理后 - 验证集NaN: 0
清理后 - 测试集NaN: 0
清理后形状 - 训练集: (79119, 142), 验证集: (19780, 142), 测试集: (9773, 142)
最终数据形状 - 训练集: (79119, 142), 验证集: (19780, 142), 测试集: (9773, 142)

2. 模型训练和优化...

--- 线性回归 ---
  RMSE: 0.2805, R²: 0.8622
  生成线性回归测试集预测...
  ✓ linear_regression预测结果已保存到: rent_predictions_linear_regression.csv
  ✓ 预测样本数量: 9773
  ✓ 预测租金范围: 32559.49 - 7116092.15

--- Lasso回归 ---
  最佳alpha: 0.001
  RMSE: 0.2775, R²: 0.8651
  生成Lasso回归测试集预测...
  ✓ lasso预测结果已保存到: rent_predictions_lasso.csv
  ✓ 预测样本数量: 9773
  ✓ 预测租金范围: 65660.08 - 7320069.42

--- 岭回归 ---
  最佳alpha: 10
  RMSE: 0.2768, R²: 0.8658
  生成岭回归测试集预测...
  ✓ ridge预测结果已保存到: rent_predictions_ridge.csv
  ✓ 预测样本数量: 9773
  ✓ 预测租金范围: 61336.57 - 7114472.52

--- 弹性网络 ---
  最佳参数: alpha=0.001, l1_ratio=0.1
  RMSE: 0.2762, R²: 0.8664
  生成弹性网络测试集预测...
  ✓ elasticnet预测结果已保存到: rent_predictions_elasticnet.csv
  ✓ 预测样本数量: 9773
  ✓ 预测租金范围

In [ ]:

print("1. 数据质量检查和清理...")

# 确保使用正确的数据
if 'X_train_prepared_opt' in locals():
    X_train_final = X_train_prepared_opt.copy()
    X_val_final = X_val_prepared_opt.copy()
    X_test_final = X_test_prepared_opt.copy()
elif 'X_train_prepared' in locals():
    X_train_final = X_train_prepared.copy()
    X_val_final = X_val_prepared.copy()
    X_test_final = X_test_prepared.copy()
else:
    print("错误：未找到准备好的数据")
    exit()

print(f"原始数据形状 - 训练集: {X_train_final.shape}, 验证集: {X_val_final.shape}, 测试集: {X_test_final.shape}")

# 目标变量对数变换
y_train_log = np.log1p(y_train_rent)
y_val_log = np.log1p(y_val_rent)

# 简化的数据清理函数 - 保持列数不变
def simple_data_cleaning(X_train, X_val, X_test):
    """简化但可靠的数据清理"""
    print("执行简化数据清理...")
    
    # 创建副本
    X_train_clean = X_train.copy()
    X_val_clean = X_val.copy()
    X_test_clean = X_test.copy()
    
    # 1. 首先处理无穷大值
    for df in [X_train_clean, X_val_clean, X_test_clean]:
        for col in df.columns:
            if df[col].dtype in [np.float64, np.int64]:
                # 替换无穷大为NaN
                df[col] = df[col].replace([np.inf, -np.inf], np.nan)
    
    # 2. 直接使用列中位数填充NaN，保持列数不变
    for col in X_train_clean.columns:
        if X_train_clean[col].dtype in [np.float64, np.int64]:
            # 计算训练集的中位数
            median_val = X_train_clean[col].median()
            
            # 如果中位数是NaN，使用0
            if pd.isna(median_val):
                median_val = 0
                
            # 填充所有数据集的NaN
            X_train_clean[col] = X_train_clean[col].fillna(median_val)
            X_val_clean[col] = X_val_clean[col].fillna(median_val)
            X_test_clean[col] = X_test_clean[col].fillna(median_val)
    
    # 3. 验证清理结果
    train_nan_after = X_train_clean.isnull().sum().sum()
    val_nan_after = X_val_clean.isnull().sum().sum()
    test_nan_after = X_test_clean.isnull().sum().sum()
    
    print(f"清理后 - 训练集NaN: {train_nan_after}")
    print(f"清理后 - 验证集NaN: {val_nan_after}")
    print(f"清理后 - 测试集NaN: {test_nan_after}")
    print(f"清理后形状 - 训练集: {X_train_clean.shape}, 验证集: {X_val_clean.shape}, 测试集: {X_test_clean.shape}")
    
    return X_train_clean, X_val_clean, X_test_clean

# 执行简化的数据清理
X_train_clean, X_val_clean, X_test_clean = simple_data_cleaning(X_train_final, X_val_final, X_test_final)

# 转换为numpy数组用于建模
X_train_array = X_train_clean.values
X_val_array = X_val_clean.values
X_test_array = X_test_clean.values

print(f"最终数据形状 - 训练集: {X_train_array.shape}, 验证集: {X_val_array.shape}, 测试集: {X_test_array.shape}")

# 定义评估函数
def evaluate_model(model, X_train, y_train, X_val, y_val, model_name):
    """评估模型性能"""
    
    try:
        # 训练集预测
        y_train_pred = model.predict(X_train)
        train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
        train_r2 = r2_score(y_train, y_train_pred)
        
        # 验证集预测
        y_val_pred = model.predict(X_val)
        val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
        val_r2 = r2_score(y_val, y_val_pred)
        val_mae = mean_absolute_error(y_val, y_val_pred)
        
        results = {
            'model_name': model_name,
            'train_rmse': train_rmse,
            'val_rmse': val_rmse,
            'train_r2': train_r2,
            'val_r2': val_r2,
            'val_mae': val_mae
        }
        
        return results, y_val_pred
    except Exception as e:
        print(f"  评估模型时出错: {e}")
        return None, None

# 🔥 新增：生成测试集预测文件的函数
def generate_test_predictions(model, X_test, model_name, test_ids):
    """生成测试集预测并保存为CSV文件"""
    try:
        # 生成预测（对数空间）
        test_predictions_log = model.predict(X_test)
        # 转换回原始空间
        test_predictions = np.expm1(test_predictions_log)
        
        # 确保预测值是合理的
        if np.any(np.isnan(test_predictions)) or np.any(np.isinf(test_predictions)):
            print(f"  警告：{model_name}预测结果包含NaN或无穷大值，进行清理")
            test_predictions = np.nan_to_num(test_predictions, nan=np.nanmedian(test_predictions))
            test_predictions = np.clip(test_predictions, test_predictions.min(), test_predictions.max())
        
        # 创建提交文件
        submission_df = pd.DataFrame({
            'ID': test_ids,
            'Rent_Prediction': test_predictions
        })
        
        # 生成文件名（去掉空格）
        filename = f"rent_predictions_{model_name.replace(' ', '_').lower()}.csv"
        submission_df.to_csv(filename, index=False)
        
        print(f"  ✓ {model_name}预测结果已保存到: {filename}")
        print(f"  ✓ 预测样本数量: {len(test_predictions)}")
        print(f"  ✓ 预测租金范围: {test_predictions.min():.2f} - {test_predictions.max():.2f}")
        
        return test_predictions
        
    except Exception as e:
        print(f"  {model_name}测试集预测出错: {e}")
        return None

print("\n2. 模型训练和优化...")

models_results = []
trained_models = {}  # 🔥 新增：存储所有训练好的模型

print("\n--- 线性回归 ---")
try:
    linear_model = LinearRegression()
    linear_model.fit(X_train_array, y_train_log)
    linear_results, linear_pred = evaluate_model(linear_model, X_train_array, y_train_log, X_val_array, y_val_log, "Linear Regression")
    if linear_results:
        models_results.append(linear_results)
        trained_models['linear'] = linear_model  # 🔥 存储模型
        print(f"  RMSE: {linear_results['val_rmse']:.4f}, R²: {linear_results['val_r2']:.4f}")
        
        # 🔥 新增：立即生成线性回归的测试集预测
        print("  生成线性回归测试集预测...")
        generate_test_predictions(linear_model, X_test_array, "linear_regression", test_ids_rent)
    else:
        print("  线性回归训练失败")
except Exception as e:
    print(f"  线性回归训练出错: {e}")

print("\n--- Lasso回归 ---")
try:
    lasso_params = {'alpha': [0.001, 0.01, 0.1, 1, 10]}
    lasso_grid = GridSearchCV(Lasso(random_state=111, max_iter=1000), lasso_params, 
                              cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
    lasso_grid.fit(X_train_array, y_train_log)
    lasso_model = lasso_grid.best_estimator_
    lasso_results, lasso_pred = evaluate_model(lasso_model, X_train_array, y_train_log, X_val_array, y_val_log, "Lasso")
    if lasso_results:
        models_results.append(lasso_results)
        trained_models['lasso'] = lasso_model  # 🔥 存储模型
        print(f"  最佳alpha: {lasso_grid.best_params_['alpha']}")
        print(f"  RMSE: {lasso_results['val_rmse']:.4f}, R²: {lasso_results['val_r2']:.4f}")
        
        # 🔥 新增：立即生成Lasso回归的测试集预测
        print("  生成Lasso回归测试集预测...")
        generate_test_predictions(lasso_model, X_test_array, "lasso", test_ids_rent)
    else:
        print("  Lasso训练失败")
except Exception as e:
    print(f"  Lasso训练出错: {e}")

print("\n--- 岭回归 ---")
try:
    ridge_params = {'alpha': [0.001, 0.01, 0.1, 1, 10]}
    ridge_grid = GridSearchCV(Ridge(random_state=111), ridge_params, 
                             cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
    ridge_grid.fit(X_train_array, y_train_log)
    ridge_model = ridge_grid.best_estimator_
    ridge_results, ridge_pred = evaluate_model(ridge_model, X_train_array, y_train_log, X_val_array, y_val_log, "Ridge")
    if ridge_results:
        models_results.append(ridge_results)
        trained_models['ridge'] = ridge_model  # 🔥 存储模型
        print(f"  最佳alpha: {ridge_grid.best_params_['alpha']}")
        print(f"  RMSE: {ridge_results['val_rmse']:.4f}, R²: {ridge_results['val_r2']:.4f}")
        
        # 🔥 新增：立即生成岭回归的测试集预测
        print("  生成岭回归测试集预测...")
        generate_test_predictions(ridge_model, X_test_array, "ridge", test_ids_rent)
    else:
        print("  岭回归训练失败")
except Exception as e:
    print(f"  岭回归训练出错: {e}")

print("\n--- 弹性网络 ---")
try:
    elastic_params = {
        'alpha': [0.001, 0.01, 0.1],
        'l1_ratio': [0.1, 0.5, 0.9]
    }
    elastic_grid = GridSearchCV(ElasticNet(random_state=111, max_iter=1000), elastic_params,
                               cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
    elastic_grid.fit(X_train_array, y_train_log)
    elastic_model = elastic_grid.best_estimator_
    elastic_results, elastic_pred = evaluate_model(elastic_model, X_train_array, y_train_log, X_val_array, y_val_log, "ElasticNet")
    if elastic_results:
        models_results.append(elastic_results)
        trained_models['elasticnet'] = elastic_model  # 🔥 存储模型
        print(f"  最佳参数: alpha={elastic_grid.best_params_['alpha']}, l1_ratio={elastic_grid.best_params_['l1_ratio']}")
        print(f"  RMSE: {elastic_results['val_rmse']:.4f}, R²: {elastic_results['val_r2']:.4f}")
        
        # 🔥 新增：立即生成弹性网络的测试集预测
        print("  生成弹性网络测试集预测...")
        generate_test_predictions(elastic_model, X_test_array, "elasticnet", test_ids_rent)
    else:
        print("  弹性网络训练失败")
except Exception as e:
    print(f"  弹性网络训练出错: {e}")


# 检查是否有成功的模型
if not models_results:
    print("所有模型训练都失败了！")
    exit()

print("\n3. 模型比较和分析...")

# 创建结果比较表格
results_df = pd.DataFrame(models_results)
results_df = results_df.sort_values('val_rmse')

print("\n" + "="*80)
print("模型性能比较")
print("="*80)
print(results_df.round(4))

# 找到最佳模型
best_model_idx = results_df['val_rmse'].idxmin()
best_model_name = results_df.loc[best_model_idx, 'model_name']

# 获取最佳模型
model_mapping = {
    'Linear Regression': trained_models.get('linear'),
    'Lasso': trained_models.get('lasso'),
    'Ridge': trained_models.get('ridge'),
    'ElasticNet': trained_models.get('elasticnet'),
}

best_model = model_mapping.get(best_model_name)

if best_model is None:
    print(f"警告：无法找到最佳模型 {best_model_name}，使用第一个可用模型")
    best_model_name = results_df.iloc[0]['model_name']
    best_model = model_mapping.get(best_model_name)

print(f"\n🏆 最佳模型: {best_model_name}")
print(f"   验证集RMSE: {results_df.loc[best_model_idx, 'val_rmse']:.4f}")
print(f"   验证集R²: {results_df.loc[best_model_idx, 'val_r2']:.4f}")
print(f"   验证集MAE: {results_df.loc[best_model_idx, 'val_mae']:.4f}")

print("\n4. 生成最佳模型的测试集预测结果...")

# 使用最佳模型进行测试集预测（作为主要提交文件）
try:
    test_predictions_log = best_model.predict(X_test_array)
    test_predictions = np.expm1(test_predictions_log)
    
    # 确保预测值是合理的
    if np.any(np.isnan(test_predictions)) or np.any(np.isinf(test_predictions)):
        print("警告：预测结果包含NaN或无穷大值，进行清理")
        test_predictions = np.nan_to_num(test_predictions, nan=np.nanmedian(test_predictions))
        test_predictions = np.clip(test_predictions, test_predictions.min(), test_predictions.max())
    
    # 创建提交文件
    submission_df = pd.DataFrame({
        'ID': test_ids_rent,
        'Rent_Prediction': test_predictions
    })
    
    # 保存预测结果（最佳模型）
    submission_df.to_csv('rent_final_predictions_best.csv', index=False)
    
    print(f"✓ 最佳模型预测结果已保存到: rent_final_predictions_best.csv")
    print(f"✓ 预测样本数量: {len(test_predictions)}")
    print(f"✓ 预测租金范围: {test_predictions.min():.2f} - {test_predictions.max():.2f}")
    
except Exception as e:
    print(f"测试集预测出错: {e}")

print("\n" + "="*80)
print("🎯 租金预测建模完成!")
print("="*80)
print(f"📊 使用的特征数量: {X_train_array.shape[1]}")
print(f"🏆 最佳模型: {best_model_name}")
print(f"📈 验证集性能:")
print(f"   - RMSE: {results_df.loc[best_model_idx, 'val_rmse']:.4f}")
print(f"   - R²: {results_df.loc[best_model_idx, 'val_r2']:.4f}")
print(f"   - MAE: {results_df.loc[best_model_idx, 'val_mae']:.4f}")
print(f"💾 输出文件:")
print(f"   - 最佳模型预测: rent_final_predictions_best.csv")
print(f"   - 线性回归预测: rent_predictions_linear_regression.csv")
print(f"   - Lasso回归预测: rent_predictions_lasso.csv")
print(f"   - 岭回归预测: rent_predictions_ridge.csv")
print(f"   - 弹性网络预测: rent_predictions_elasticnet.csv")
print("="*80)

In [16]:
# --- 分别编号合并房价和租金预测结果为单个CSV文件 ---
print("\n--- 分别编号合并房价和租金预测结果为单个CSV文件 ---")

import pandas as pd
import numpy as np
import os

try:
    # 1. 读取两个预测文件
    print("读取预测文件...")
    price_df = pd.read_csv('optimized_price_prediction.csv')
    rent_df = pd.read_csv('rent_final_predictions.csv')
    
    print(f"房价预测文件: {price_df.shape}")
    print(f"租金预测文件: {rent_df.shape}")
    
    # 2. 检查两个文件的结构
    print("\n房价预测文件列名:", price_df.columns.tolist())
    print("租金预测文件列名:", rent_df.columns.tolist())
    
    # 3. 为房价和租金分别创建不同的ID范围
    # 房价ID从1000000开始，租金ID从2000000开始
    price_start_id = 1000000
    rent_start_id = 2000000
    
    # 为房价数据创建新ID
    price_df_renamed = price_df.rename(columns={'Price': 'Prediction'})
    price_df_renamed['ID'] = range(price_start_id, price_start_id + len(price_df_renamed))
    
    # 为租金数据创建新ID
    rent_df_renamed = rent_df.rename(columns={'Rent_Prediction': 'Prediction'})
    rent_df_renamed['ID'] = range(rent_start_id, rent_start_id + len(rent_df_renamed))
    
    # 只保留ID和Prediction列
    price_df_final = price_df_renamed[['ID', 'Prediction']]
    rent_df_final = rent_df_renamed[['ID', 'Prediction']]
    
    # 4. 上下合并两个文件
    print("\n上下合并文件...")
    merged_df = pd.concat([price_df_final, rent_df_final], axis=0, ignore_index=True)
    
    print(f"合并后数据形状: {merged_df.shape}")
    print(f"合并后列名: {merged_df.columns.tolist()}")
    
    # 5. 检查合并结果
    print("\n合并结果统计:")
    print(f"总记录数: {len(merged_df)}")
    print(f"来自房价预测的记录数: {len(price_df)}")
    print(f"来自租金预测的记录数: {len(rent_df)}")
    print(f"房价ID范围: {price_start_id} - {price_start_id + len(price_df) - 1}")
    print(f"租金ID范围: {rent_start_id} - {rent_start_id + len(rent_df) - 1}")
    
    # 6. 检查是否有缺失值并处理
    if merged_df.isnull().any().any():
        print("\n发现缺失值，进行处理...")
        missing_pred = merged_df['Prediction'].isnull().sum()
        
        print(f"缺失Prediction值: {missing_pred}")
        
        if missing_pred > 0:
            # 用中位数填充缺失预测值
            pred_median = merged_df['Prediction'].median()
            merged_df['Prediction'] = merged_df['Prediction'].fillna(pred_median)
            print(f"Prediction缺失值已用中位数 {pred_median:.2f} 填充")
    
    # 7. 确保数据格式正确
    merged_df['ID'] = merged_df['ID'].astype(int)
    merged_df['Prediction'] = merged_df['Prediction'].round(2)  # 保留两位小数
    
    # 8. 按ID排序
    merged_df = merged_df.sort_values('ID')
    
    # 9. 保存合并后的CSV文件
    output_filename = 'kaggle_submission_final.csv'
    
    # 检查文件是否已存在，如果存在则删除
    if os.path.exists(output_filename):
        try:
            os.remove(output_filename)
            print(f"已删除已存在的文件: {output_filename}")
        except Exception as e:
            print(f"无法删除文件 {output_filename}: {e}")
            # 尝试使用另一个文件名
            output_filename = 'submission_' + str(pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')) + '.csv'
            print(f"将使用新文件名: {output_filename}")
    
    # 保存文件
    merged_df.to_csv(output_filename, index=False)
    
    # 10. 验证输出文件
    print(f"\n合并完成! 输出文件: {output_filename}")
    print(f"最终数据形状: {merged_df.shape}")
    
    # 显示一些统计信息
    print("\n最终数据统计:")
    print(f"ID范围: {merged_df['ID'].min()} - {merged_df['ID'].max()}")
    print(f"Prediction统计:")
    print(f"  最小值: {merged_df['Prediction'].min():.2f}")
    print(f"  最大值: {merged_df['Prediction'].max():.2f}")
    print(f"  平均值: {merged_df['Prediction'].mean():.2f}")
    print(f"  中位数: {merged_df['Prediction'].median():.2f}")
    
    # 显示前几行作为预览
    print(f"\n前10行房价数据预览:")
    price_preview = merged_df[merged_df['ID'] < 2000000].head(10)
    print(price_preview.to_string(index=False))
    
    print(f"\n前10行租金数据预览:")
    rent_preview = merged_df[merged_df['ID'] >= 2000000].head(10)
    print(rent_preview.to_string(index=False))
    
    print(f"\n🎉 分别编号合并完成！文件已保存为: {output_filename}")
    print("您现在可以将此CSV文件提交到Kaggle进行测评")

except FileNotFoundError as e:
    print(f"文件未找到错误: {e}")
    print("请确保以下文件存在于当前目录:")
    print("  - optimized_price_prediction.csv")
    print("  - rent_final_predictions.csv")
    
    # 尝试查找可能的文件名变体
    import glob
    csv_files = glob.glob("*.csv")
    if csv_files:
        print("\n当前目录中的CSV文件:")
        for file in csv_files:
            print(f"  - {file}")
    
except PermissionError as e:
    print(f"权限错误: {e}")
    print("请关闭可能正在使用该文件的程序，然后重试")
    
except Exception as e:
    print(f"合并过程中发生错误: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*80)
print("CSV文件分别编号合并完成!")
print("="*80)


--- 分别编号合并房价和租金预测结果为单个CSV文件 ---
读取预测文件...
房价预测文件: (34017, 2)
租金预测文件: (9773, 2)

房价预测文件列名: ['ID', 'Price']
租金预测文件列名: ['ID', 'Rent_Prediction']

上下合并文件...
合并后数据形状: (43790, 2)
合并后列名: ['ID', 'Prediction']

合并结果统计:
总记录数: 43790
来自房价预测的记录数: 34017
来自租金预测的记录数: 9773
房价ID范围: 1000000 - 1034016
租金ID范围: 2000000 - 2009772
已删除已存在的文件: kaggle_submission_final.csv

合并完成! 输出文件: kaggle_submission_final.csv
最终数据形状: (43790, 2)

最终数据统计:
ID范围: 1000000 - 2009772
Prediction统计:
  最小值: 67828.37
  最大值: 29257032.88
  平均值: 2014864.87
  中位数: 1348068.28

前10行房价数据预览:
     ID  Prediction
1000000 20177261.72
1000001  2608389.55
1000002  4975103.06
1000003  3325657.30
1000004  8676035.91
1000005  3071109.70
1000006 16874390.68
1000007  4387188.05
1000008  5191745.96
1000009  9465196.89

前10行租金数据预览:
     ID  Prediction
2000000   170518.25
2000001   372236.96
2000002   425630.20
2000003  1662647.77
2000004  1180090.46
2000005   336633.33
2000006   326200.66
2000007   794495.71
2000008   246957.37
2000009   152505.84

🎉 分别编号

以下代码为添加交互项的最优模型合并，仅为输出对比kaggle得分，在模型建立中无实际意义

In [17]:
# --- 分别编号合并房价和租金预测结果为单个CSV文件 ---
print("\n--- 分别编号合并房价和租金预测结果为单个CSV文件 ---")

import pandas as pd
import numpy as np
import os

try:
    # 1. 读取两个预测文件
    print("读取预测文件...")
    price_df = pd.read_csv('optimized_price_prediction_with_interactions.csv')
    rent_df = pd.read_csv('rent_final_predictions.csv')
    
    print(f"房价预测文件: {price_df.shape}")
    print(f"租金预测文件: {rent_df.shape}")
    
    # 2. 检查两个文件的结构
    print("\n房价预测文件列名:", price_df.columns.tolist())
    print("租金预测文件列名:", rent_df.columns.tolist())
    
    # 3. 为房价和租金分别创建不同的ID范围
    # 房价ID从1000000开始，租金ID从2000000开始
    price_start_id = 1000000
    rent_start_id = 2000000
    
    # 为房价数据创建新ID
    price_df_renamed = price_df.rename(columns={'Price': 'Prediction'})
    price_df_renamed['ID'] = range(price_start_id, price_start_id + len(price_df_renamed))
    
    # 为租金数据创建新ID
    rent_df_renamed = rent_df.rename(columns={'Rent_Prediction': 'Prediction'})
    rent_df_renamed['ID'] = range(rent_start_id, rent_start_id + len(rent_df_renamed))
    
    # 只保留ID和Prediction列
    price_df_final = price_df_renamed[['ID', 'Prediction']]
    rent_df_final = rent_df_renamed[['ID', 'Prediction']]
    
    # 4. 上下合并两个文件
    print("\n上下合并文件...")
    merged_df = pd.concat([price_df_final, rent_df_final], axis=0, ignore_index=True)
    
    print(f"合并后数据形状: {merged_df.shape}")
    print(f"合并后列名: {merged_df.columns.tolist()}")
    
    # 5. 检查合并结果
    print("\n合并结果统计:")
    print(f"总记录数: {len(merged_df)}")
    print(f"来自房价预测的记录数: {len(price_df)}")
    print(f"来自租金预测的记录数: {len(rent_df)}")
    print(f"房价ID范围: {price_start_id} - {price_start_id + len(price_df) - 1}")
    print(f"租金ID范围: {rent_start_id} - {rent_start_id + len(rent_df) - 1}")
    
    # 6. 检查是否有缺失值并处理
    if merged_df.isnull().any().any():
        print("\n发现缺失值，进行处理...")
        missing_pred = merged_df['Prediction'].isnull().sum()
        
        print(f"缺失Prediction值: {missing_pred}")
        
        if missing_pred > 0:
            # 用中位数填充缺失预测值
            pred_median = merged_df['Prediction'].median()
            merged_df['Prediction'] = merged_df['Prediction'].fillna(pred_median)
            print(f"Prediction缺失值已用中位数 {pred_median:.2f} 填充")
    
    # 7. 确保数据格式正确
    merged_df['ID'] = merged_df['ID'].astype(int)
    merged_df['Prediction'] = merged_df['Prediction'].round(2)  # 保留两位小数
    
    # 8. 按ID排序
    merged_df = merged_df.sort_values('ID')
    
    # 9. 保存合并后的CSV文件
    output_filename = 'kaggle_submission_final_with_interactions.csv'
    
    # 检查文件是否已存在，如果存在则删除
    if os.path.exists(output_filename):
        try:
            os.remove(output_filename)
            print(f"已删除已存在的文件: {output_filename}")
        except Exception as e:
            print(f"无法删除文件 {output_filename}: {e}")
            # 尝试使用另一个文件名
            output_filename = 'submission_with_interactions_' + str(pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')) + '.csv'
            print(f"将使用新文件名: {output_filename}")
    
    # 保存文件
    merged_df.to_csv(output_filename, index=False)
    
    # 10. 验证输出文件
    print(f"\n合并完成! 输出文件: {output_filename}")
    print(f"最终数据形状: {merged_df.shape}")
    
    # 显示一些统计信息
    print("\n最终数据统计:")
    print(f"ID范围: {merged_df['ID'].min()} - {merged_df['ID'].max()}")
    print(f"Prediction统计:")
    print(f"  最小值: {merged_df['Prediction'].min():.2f}")
    print(f"  最大值: {merged_df['Prediction'].max():.2f}")
    print(f"  平均值: {merged_df['Prediction'].mean():.2f}")
    print(f"  中位数: {merged_df['Prediction'].median():.2f}")
    
    # 显示前几行作为预览
    print(f"\n前10行房价数据预览:")
    price_preview = merged_df[merged_df['ID'] < 2000000].head(10)
    print(price_preview.to_string(index=False))
    
    print(f"\n前10行租金数据预览:")
    rent_preview = merged_df[merged_df['ID'] >= 2000000].head(10)
    print(rent_preview.to_string(index=False))
    
    print(f"\n🎉 分别编号合并完成！文件已保存为: {output_filename}")
    print("您现在可以将此CSV文件提交到Kaggle进行测评")

except FileNotFoundError as e:
    print(f"文件未找到错误: {e}")
    print("请确保以下文件存在于当前目录:")
    print("  - optimized_price_prediction_with_interactions.csv")
    print("  - rent_final_predictions.csv")
    
    # 尝试查找可能的文件名变体
    import glob
    csv_files = glob.glob("*.csv")
    if csv_files:
        print("\n当前目录中的CSV文件:")
        for file in csv_files:
            print(f"  - {file}")
    
except PermissionError as e:
    print(f"权限错误: {e}")
    print("请关闭可能正在使用该文件的程序，然后重试")
    
except Exception as e:
    print(f"合并过程中发生错误: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*80)
print("CSV文件分别编号合并完成!")
print("="*80)


--- 分别编号合并房价和租金预测结果为单个CSV文件 ---
读取预测文件...
房价预测文件: (34017, 2)
租金预测文件: (9773, 2)

房价预测文件列名: ['ID', 'Price']
租金预测文件列名: ['ID', 'Rent_Prediction']

上下合并文件...
合并后数据形状: (43790, 2)
合并后列名: ['ID', 'Prediction']

合并结果统计:
总记录数: 43790
来自房价预测的记录数: 34017
来自租金预测的记录数: 9773
房价ID范围: 1000000 - 1034016
租金ID范围: 2000000 - 2009772
已删除已存在的文件: kaggle_submission_final_with_interactions.csv

合并完成! 输出文件: kaggle_submission_final_with_interactions.csv
最终数据形状: (43790, 2)

最终数据统计:
ID范围: 1000000 - 2009772
Prediction统计:
  最小值: 67828.37
  最大值: 30665254.59
  平均值: 2012718.93
  中位数: 1332038.63

前10行房价数据预览:
     ID  Prediction
1000000 21979350.71
1000001  2681210.12
1000002  4604846.41
1000003  3324222.19
1000004  8802489.26
1000005  3084522.24
1000006 16514900.31
1000007  4396380.00
1000008  5246736.68
1000009  9477426.33

前10行租金数据预览:
     ID  Prediction
2000000   170518.25
2000001   372236.96
2000002   425630.20
2000003  1662647.77
2000004  1180090.46
2000005   336633.33
2000006   326200.66
2000007   794495.71
2000008   2

以下仅为合并相应线性模型的输出用于kaggle测评，无实际编码意义

In [20]:
# --- 分别编号合并四个线性模型的房价和租金预测结果为单个CSV文件 ---
print("\n--- 分别编号合并四个线性模型的房价和租金预测结果为单个CSV文件 ---")

import pandas as pd
import numpy as np
import os

# 定义模型名称和对应的文件
model_files = {
    'linear_regression': {
        'price': 'linear_regression_predictions.csv',
        'rent': 'rent_predictions_linear_regression.csv'
    },
    'lasso': {
        'price': 'lasso_predictions.csv', 
        'rent': 'rent_predictions_lasso.csv'
    },
    'ridge': {
        'price': 'ridge_predictions.csv',
        'rent': 'rent_predictions_ridge.csv'
    },
    'elasticnet': {
        'price': 'elasticnet_predictions.csv',
        'rent': 'rent_predictions_elasticnet.csv'
    }
}

try:
    # 为每个模型创建合并文件
    for model_name, files in model_files.items():
        print(f"\n{'='*60}")
        print(f"处理 {model_name} 模型...")
        print(f"{'='*60}")
        
        # 1. 读取两个预测文件
        print("读取预测文件...")
        try:
            price_df = pd.read_csv(files['price'])
            rent_df = pd.read_csv(files['rent'])
            
            print(f"房价预测文件: {price_df.shape}")
            print(f"租金预测文件: {rent_df.shape}")
            
        except FileNotFoundError as e:
            print(f"文件未找到: {e}")
            print(f"请确保以下文件存在:")
            print(f"  - {files['price']}")
            print(f"  - {files['rent']}")
            continue
        
        # 2. 检查两个文件的结构
        print("\n房价预测文件列名:", price_df.columns.tolist())
        print("租金预测文件列名:", rent_df.columns.tolist())
        
        # 3. 为房价和租金分别创建不同的ID范围
        # 房价ID从1000000开始，租金ID从2000000开始
        price_start_id = 1000000
        rent_start_id = 2000000
        
        # 为房价数据创建新ID
        # 检查房价文件的列名并重命名
        if 'Price' in price_df.columns:
            price_df_renamed = price_df.rename(columns={'Price': 'Prediction'})
        elif 'Prediction' in price_df.columns:
            price_df_renamed = price_df.copy()
        else:
            print(f"错误：房价文件 {files['price']} 的列名不符合预期")
            continue
            
        price_df_renamed['ID'] = range(price_start_id, price_start_id + len(price_df_renamed))
        
        # 为租金数据创建新ID
        # 检查租金文件的列名并重命名
        if 'Rent_Prediction' in rent_df.columns:
            rent_df_renamed = rent_df.rename(columns={'Rent_Prediction': 'Prediction'})
        elif 'Prediction' in rent_df.columns:
            rent_df_renamed = rent_df.copy()
        else:
            print(f"错误：租金文件 {files['rent']} 的列名不符合预期")
            continue
            
        rent_df_renamed['ID'] = range(rent_start_id, rent_start_id + len(rent_df_renamed))
        
        # 只保留ID和Prediction列
        price_df_final = price_df_renamed[['ID', 'Prediction']]
        rent_df_final = rent_df_renamed[['ID', 'Prediction']]
        
        # 4. 上下合并两个文件
        print("\n上下合并文件...")
        merged_df = pd.concat([price_df_final, rent_df_final], axis=0, ignore_index=True)
        
        print(f"合并后数据形状: {merged_df.shape}")
        print(f"合并后列名: {merged_df.columns.tolist()}")
        
        # 5. 检查合并结果
        print("\n合并结果统计:")
        print(f"总记录数: {len(merged_df)}")
        print(f"来自房价预测的记录数: {len(price_df)}")
        print(f"来自租金预测的记录数: {len(rent_df)}")
        print(f"房价ID范围: {price_start_id} - {price_start_id + len(price_df) - 1}")
        print(f"租金ID范围: {rent_start_id} - {rent_start_id + len(rent_df) - 1}")
        
        # 6. 检查是否有缺失值并处理
        if merged_df.isnull().any().any():
            print("\n发现缺失值，进行处理...")
            missing_pred = merged_df['Prediction'].isnull().sum()
            
            print(f"缺失Prediction值: {missing_pred}")
            
            if missing_pred > 0:
                # 用中位数填充缺失预测值
                pred_median = merged_df['Prediction'].median()
                merged_df['Prediction'] = merged_df['Prediction'].fillna(pred_median)
                print(f"Prediction缺失值已用中位数 {pred_median:.2f} 填充")
        
        # 7. 确保数据格式正确
        merged_df['ID'] = merged_df['ID'].astype(int)
        merged_df['Prediction'] = merged_df['Prediction'].round(2)  # 保留两位小数
        
        # 8. 按ID排序
        merged_df = merged_df.sort_values('ID')
        
        # 9. 保存合并后的CSV文件
        output_filename = f'kaggle_submission_{model_name}.csv'
        
        # 检查文件是否已存在，如果存在则删除
        if os.path.exists(output_filename):
            try:
                os.remove(output_filename)
                print(f"已删除已存在的文件: {output_filename}")
            except Exception as e:
                print(f"无法删除文件 {output_filename}: {e}")
                # 尝试使用另一个文件名
                output_filename = f'submission_{model_name}_' + str(pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')) + '.csv'
                print(f"将使用新文件名: {output_filename}")
        
        # 保存文件
        merged_df.to_csv(output_filename, index=False)
        
        # 10. 验证输出文件
        print(f"\n合并完成! 输出文件: {output_filename}")
        print(f"最终数据形状: {merged_df.shape}")
        
        # 显示一些统计信息
        print("\n最终数据统计:")
        print(f"ID范围: {merged_df['ID'].min()} - {merged_df['ID'].max()}")
        print(f"Prediction统计:")
        print(f"  最小值: {merged_df['Prediction'].min():.2f}")
        print(f"  最大值: {merged_df['Prediction'].max():.2f}")
        print(f"  平均值: {merged_df['Prediction'].mean():.2f}")
        print(f"  中位数: {merged_df['Prediction'].median():.2f}")
        
        # 显示前几行作为预览
        print(f"\n前5行房价数据预览:")
        price_preview = merged_df[merged_df['ID'] < 2000000].head(5)
        print(price_preview.to_string(index=False))
        
        print(f"\n前5行租金数据预览:")
        rent_preview = merged_df[merged_df['ID'] >= 2000000].head(5)
        print(rent_preview.to_string(index=False))
        
        print(f"\n✅ {model_name} 模型合并完成！文件已保存为: {output_filename}")

    print(f"\n{'='*80}")
    print("🎉 所有模型分别编号合并完成！")
    print("生成的Kaggle提交文件:")
    for model_name in model_files.keys():
        print(f"  - kaggle_submission_{model_name}.csv")
    print("您现在可以将这些CSV文件提交到Kaggle进行测评")
    print("="*80)

except Exception as e:
    print(f"合并过程中发生错误: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*80)
print("四个线性模型的CSV文件分别编号合并完成!")
print("="*80)


--- 分别编号合并四个线性模型的房价和租金预测结果为单个CSV文件 ---

处理 linear_regression 模型...
读取预测文件...
房价预测文件: (34017, 2)
租金预测文件: (9773, 2)

房价预测文件列名: ['ID', 'Price']
租金预测文件列名: ['ID', 'Rent_Prediction']

上下合并文件...
合并后数据形状: (43790, 2)
合并后列名: ['ID', 'Prediction']

合并结果统计:
总记录数: 43790
来自房价预测的记录数: 34017
来自租金预测的记录数: 9773
房价ID范围: 1000000 - 1034016
租金ID范围: 2000000 - 2009772

合并完成! 输出文件: kaggle_submission_linear_regression.csv
最终数据形状: (43790, 2)

最终数据统计:
ID范围: 1000000 - 2009772
Prediction统计:
  最小值: 32559.49
  最大值: 29254041.41
  平均值: 2011757.02
  中位数: 1347622.24

前5行房价数据预览:
     ID  Prediction
1000000 20175450.14
1000001  2608442.73
1000002  4975296.43
1000003  3325923.55
1000004  8676192.10

前5行租金数据预览:
     ID  Prediction
2000000   182195.44
2000001   364651.27
2000002   443892.18
2000003  1591971.02
2000004  1210728.14

✅ linear_regression 模型合并完成！文件已保存为: kaggle_submission_linear_regression.csv

处理 lasso 模型...
读取预测文件...
房价预测文件: (34017, 2)
租金预测文件: (9773, 2)

房价预测文件列名: ['ID', 'Price']
租金预测文件列名: ['ID', 'Rent_Prediction']

仅添加交叉验证和得到的kaggle得分为输出所需表格，由于在原输出代码上添加有之前有很大重复性，可直接看表格结果

In [22]:
# --- 租金预测建模与评估 ---
print("\n--- 租金预测建模与评估 ---")

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("1. 数据质量检查和清理...")

# 确保使用正确的数据
if 'X_train_prepared_opt' in locals():
    X_train_final = X_train_prepared_opt.copy()
    X_val_final = X_val_prepared_opt.copy()
    X_test_final = X_test_prepared_opt.copy()
elif 'X_train_prepared' in locals():
    X_train_final = X_train_prepared.copy()
    X_val_final = X_val_prepared.copy()
    X_test_final = X_test_prepared.copy()
else:
    print("错误：未找到准备好的数据")
    exit()

print(f"原始数据形状 - 训练集: {X_train_final.shape}, 验证集: {X_val_final.shape}, 测试集: {X_test_final.shape}")

# 目标变量对数变换
y_train_log = np.log1p(y_train_rent)
y_val_log = np.log1p(y_val_rent)

# 简化的数据清理函数 - 保持列数不变
def simple_data_cleaning(X_train, X_val, X_test):
    """简化但可靠的数据清理"""
    print("执行简化数据清理...")
    
    # 创建副本
    X_train_clean = X_train.copy()
    X_val_clean = X_val.copy()
    X_test_clean = X_test.copy()
    
    # 1. 首先处理无穷大值
    for df in [X_train_clean, X_val_clean, X_test_clean]:
        for col in df.columns:
            if df[col].dtype in [np.float64, np.int64]:
                # 替换无穷大为NaN
                df[col] = df[col].replace([np.inf, -np.inf], np.nan)
    
    # 2. 直接使用列中位数填充NaN，保持列数不变
    for col in X_train_clean.columns:
        if X_train_clean[col].dtype in [np.float64, np.int64]:
            # 计算训练集的中位数
            median_val = X_train_clean[col].median()
            
            # 如果中位数是NaN，使用0
            if pd.isna(median_val):
                median_val = 0
                
            # 填充所有数据集的NaN
            X_train_clean[col] = X_train_clean[col].fillna(median_val)
            X_val_clean[col] = X_val_clean[col].fillna(median_val)
            X_test_clean[col] = X_test_clean[col].fillna(median_val)
    
    # 3. 验证清理结果
    train_nan_after = X_train_clean.isnull().sum().sum()
    val_nan_after = X_val_clean.isnull().sum().sum()
    test_nan_after = X_test_clean.isnull().sum().sum()
    
    print(f"清理后 - 训练集NaN: {train_nan_after}")
    print(f"清理后 - 验证集NaN: {val_nan_after}")
    print(f"清理后 - 测试集NaN: {test_nan_after}")
    print(f"清理后形状 - 训练集: {X_train_clean.shape}, 验证集: {X_val_clean.shape}, 测试集: {X_test_clean.shape}")
    
    return X_train_clean, X_val_clean, X_test_clean

# 执行简化的数据清理
X_train_clean, X_val_clean, X_test_clean = simple_data_cleaning(X_train_final, X_val_final, X_test_final)

# 转换为numpy数组用于建模
X_train_array = X_train_clean.values
X_val_array = X_val_clean.values
X_test_array = X_test_clean.values

print(f"最终数据形状 - 训练集: {X_train_array.shape}, 验证集: {X_val_array.shape}, 测试集: {X_test_array.shape}")

# 🔥 修改：增强评估函数，添加交叉验证
def evaluate_model_with_cv(model, X_train, y_train, X_val, y_val, model_name, cv_folds=5):
    """评估模型性能，包括交叉验证"""
    
    try:
        # 训练集预测
        y_train_pred = model.predict(X_train)
        train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
        train_r2 = r2_score(y_train, y_train_pred)
        
        # 验证集预测
        y_val_pred = model.predict(X_val)
        val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
        val_r2 = r2_score(y_val, y_val_pred)
        val_mae = mean_absolute_error(y_val, y_val_pred)
        
        # 🔥 新增：交叉验证R²
        cv_scores = cross_val_score(model, X_train, y_train, cv=cv_folds, scoring='r2')
        cv_r2_mean = cv_scores.mean()
        cv_r2_std = cv_scores.std()
        
        results = {
            'model_name': model_name,
            'train_rmse': train_rmse,
            'val_rmse': val_rmse,
            'train_r2': train_r2,
            'val_r2': val_r2,
            'val_mae': val_mae,
            'cv_r2_mean': cv_r2_mean,  # 🔥 新增
            'cv_r2_std': cv_r2_std,    # 🔥 新增
            'cv_folds': cv_folds        # 🔥 新增
        }
        
        return results, y_val_pred, cv_scores
    except Exception as e:
        print(f"  评估模型时出错: {e}")
        return None, None, None

# 🔥 新增：生成测试集预测文件的函数
def generate_test_predictions(model, X_test, model_name, test_ids):
    """生成测试集预测并保存为CSV文件"""
    try:
        # 生成预测（对数空间）
        test_predictions_log = model.predict(X_test)
        # 转换回原始空间
        test_predictions = np.expm1(test_predictions_log)
        
        # 确保预测值是合理的
        if np.any(np.isnan(test_predictions)) or np.any(np.isinf(test_predictions)):
            print(f"  警告：{model_name}预测结果包含NaN或无穷大值，进行清理")
            test_predictions = np.nan_to_num(test_predictions, nan=np.nanmedian(test_predictions))
            test_predictions = np.clip(test_predictions, test_predictions.min(), test_predictions.max())
        
        # 创建提交文件
        submission_df = pd.DataFrame({
            'ID': test_ids,
            'Rent_Prediction': test_predictions
        })
        
        # 生成文件名（去掉空格）
        filename = f"rent_predictions_{model_name.replace(' ', '_').lower()}.csv"
        submission_df.to_csv(filename, index=False)
        
        print(f"  ✓ {model_name}预测结果已保存到: {filename}")
        print(f"  ✓ 预测样本数量: {len(test_predictions)}")
        print(f"  ✓ 预测租金范围: {test_predictions.min():.2f} - {test_predictions.max():.2f}")
        
        return test_predictions
        
    except Exception as e:
        print(f"  {model_name}测试集预测出错: {e}")
        return None

print("\n2. 模型训练和优化...")

models_results = []
trained_models = {}  # 🔥 存储所有训练好的模型

print("\n--- 线性回归 ---")
try:
    linear_model = LinearRegression()
    linear_model.fit(X_train_array, y_train_log)
    linear_results, linear_pred, linear_cv_scores = evaluate_model_with_cv(
        linear_model, X_train_array, y_train_log, X_val_array, y_val_log, "Linear Regression"
    )
    if linear_results:
        models_results.append(linear_results)
        trained_models['linear'] = linear_model
        print(f"  RMSE: {linear_results['val_rmse']:.4f}, R²: {linear_results['val_r2']:.4f}")
        print(f"  交叉验证R²: {linear_results['cv_r2_mean']:.4f} ± {linear_results['cv_r2_std']:.4f}")
        
        # 生成线性回归的测试集预测
        print("  生成线性回归测试集预测...")
        generate_test_predictions(linear_model, X_test_array, "linear_regression", test_ids_rent)
    else:
        print("  线性回归训练失败")
except Exception as e:
    print(f"  线性回归训练出错: {e}")

print("\n--- Lasso回归 ---")
try:
    lasso_params = {'alpha': [0.001, 0.01, 0.1, 1, 10]}
    lasso_grid = GridSearchCV(Lasso(random_state=111, max_iter=1000), lasso_params, 
                              cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
    lasso_grid.fit(X_train_array, y_train_log)
    lasso_model = lasso_grid.best_estimator_
    lasso_results, lasso_pred, lasso_cv_scores = evaluate_model_with_cv(
        lasso_model, X_train_array, y_train_log, X_val_array, y_val_log, "Lasso"
    )
    if lasso_results:
        models_results.append(lasso_results)
        trained_models['lasso'] = lasso_model
        print(f"  最佳alpha: {lasso_grid.best_params_['alpha']}")
        print(f"  RMSE: {lasso_results['val_rmse']:.4f}, R²: {lasso_results['val_r2']:.4f}")
        print(f"  交叉验证R²: {lasso_results['cv_r2_mean']:.4f} ± {lasso_results['cv_r2_std']:.4f}")
        
        # 生成Lasso回归的测试集预测
        print("  生成Lasso回归测试集预测...")
        generate_test_predictions(lasso_model, X_test_array, "lasso", test_ids_rent)
    else:
        print("  Lasso训练失败")
except Exception as e:
    print(f"  Lasso训练出错: {e}")

print("\n--- 岭回归 ---")
try:
    ridge_params = {'alpha': [0.001, 0.01, 0.1, 1, 10]}
    ridge_grid = GridSearchCV(Ridge(random_state=111), ridge_params, 
                             cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
    ridge_grid.fit(X_train_array, y_train_log)
    ridge_model = ridge_grid.best_estimator_
    ridge_results, ridge_pred, ridge_cv_scores = evaluate_model_with_cv(
        ridge_model, X_train_array, y_train_log, X_val_array, y_val_log, "Ridge"
    )
    if ridge_results:
        models_results.append(ridge_results)
        trained_models['ridge'] = ridge_model
        print(f"  最佳alpha: {ridge_grid.best_params_['alpha']}")
        print(f"  RMSE: {ridge_results['val_rmse']:.4f}, R²: {ridge_results['val_r2']:.4f}")
        print(f"  交叉验证R²: {ridge_results['cv_r2_mean']:.4f} ± {ridge_results['cv_r2_std']:.4f}")
        
        # 生成岭回归的测试集预测
        print("  生成岭回归测试集预测...")
        generate_test_predictions(ridge_model, X_test_array, "ridge", test_ids_rent)
    else:
        print("  岭回归训练失败")
except Exception as e:
    print(f"  岭回归训练出错: {e}")

print("\n--- 弹性网络 ---")
try:
    elastic_params = {
        'alpha': [0.001, 0.01, 0.1],
        'l1_ratio': [0.1, 0.5, 0.9]
    }
    elastic_grid = GridSearchCV(ElasticNet(random_state=111, max_iter=1000), elastic_params,
                               cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
    elastic_grid.fit(X_train_array, y_train_log)
    elastic_model = elastic_grid.best_estimator_
    elastic_results, elastic_pred, elastic_cv_scores = evaluate_model_with_cv(
        elastic_model, X_train_array, y_train_log, X_val_array, y_val_log, "ElasticNet"
    )
    if elastic_results:
        models_results.append(elastic_results)
        trained_models['elasticnet'] = elastic_model
        print(f"  最佳参数: alpha={elastic_grid.best_params_['alpha']}, l1_ratio={elastic_grid.best_params_['l1_ratio']}")
        print(f"  RMSE: {elastic_results['val_rmse']:.4f}, R²: {elastic_results['val_r2']:.4f}")
        print(f"  交叉验证R²: {elastic_results['cv_r2_mean']:.4f} ± {elastic_results['cv_r2_std']:.4f}")
        
        # 生成弹性网络的测试集预测
        print("  生成弹性网络测试集预测...")
        generate_test_predictions(elastic_model, X_test_array, "elasticnet", test_ids_rent)
    else:
        print("  弹性网络训练失败")
except Exception as e:
    print(f"  弹性网络训练出错: {e}")


# 检查是否有成功的模型
if not models_results:
    print("所有模型训练都失败了！")
    exit()

print("\n3. 模型比较和分析...")

# 创建结果比较表格
results_df = pd.DataFrame(models_results)

# 🔥 新增：添加Kaggle得分
kaggle_scores = {
    'Linear Regression': 65.1,  
    'Lasso': 65.7,             
    'Ridge': 65.7,            
    'ElasticNet': 65.8        
}

results_df['kaggle_score'] = results_df['model_name'].map(kaggle_scores)

# 按验证集RMSE排序
results_df = results_df.sort_values('val_rmse')

print("\n" + "="*100)
print("租金模型综合性能比较")
print("="*100)

# 🔥 新增：生成详细的性能表格
performance_table = results_df[[
    'model_name', 'train_r2', 'val_r2', 'cv_r2_mean', 'kaggle_score', 
    'train_rmse', 'val_rmse', 'val_mae'
]].round(4)

# 重命名列名以便更好理解
performance_table = performance_table.rename(columns={
    'model_name': '模型名称',
    'train_r2': '样本内R²',
    'val_r2': '样本外R²', 
    'cv_r2_mean': '交叉验证R²',
    'kaggle_score': 'Kaggle得分',
    'train_rmse': '训练集RMSE',
    'val_rmse': '验证集RMSE',
    'val_mae': '验证集MAE'
})

print(performance_table.to_string(index=False))

# 🔥 新增：生成总结表格（只包含R²相关指标）
print("\n" + "="*80)
print("R²指标对比总结")
print("="*80)

summary_table = results_df[[
    'model_name', 'train_r2', 'val_r2', 'cv_r2_mean', 'kaggle_score'
]].round(4)

summary_table = summary_table.rename(columns={
    'model_name': '模型',
    'train_r2': '样本内R²',
    'val_r2': '样本外R²',
    'cv_r2_mean': '交叉验证R²', 
    'kaggle_score': 'Kaggle得分'
})

print(summary_table.to_string(index=False))

# 找到最佳模型
best_model_idx = results_df['val_rmse'].idxmin()
best_model_name = results_df.loc[best_model_idx, 'model_name']

# 获取最佳模型
model_mapping = {
    'Linear Regression': trained_models.get('linear'),
    'Lasso': trained_models.get('lasso'),
    'Ridge': trained_models.get('ridge'),
    'ElasticNet': trained_models.get('elasticnet'),
}

best_model = model_mapping.get(best_model_name)

if best_model is None:
    print(f"警告：无法找到最佳模型 {best_model_name}，使用第一个可用模型")
    best_model_name = results_df.iloc[0]['model_name']
    best_model = model_mapping.get(best_model_name)

print(f"\n🏆 最佳模型: {best_model_name}")
print(f"   验证集RMSE: {results_df.loc[best_model_idx, 'val_rmse']:.4f}")
print(f"   验证集R²: {results_df.loc[best_model_idx, 'val_r2']:.4f}")
print(f"   验证集MAE: {results_df.loc[best_model_idx, 'val_mae']:.4f}")

print("\n4. 生成最佳模型的测试集预测结果...")

# 使用最佳模型进行测试集预测（作为主要提交文件）
try:
    test_predictions_log = best_model.predict(X_test_array)
    test_predictions = np.expm1(test_predictions_log)
    
    # 确保预测值是合理的
    if np.any(np.isnan(test_predictions)) or np.any(np.isinf(test_predictions)):
        print("警告：预测结果包含NaN或无穷大值，进行清理")
        test_predictions = np.nan_to_num(test_predictions, nan=np.nanmedian(test_predictions))
        test_predictions = np.clip(test_predictions, test_predictions.min(), test_predictions.max())
    
    # 创建提交文件
    submission_df = pd.DataFrame({
        'ID': test_ids_rent,
        'Rent_Prediction': test_predictions
    })
    
    # 保存预测结果（最佳模型）
    submission_df.to_csv('rent_final_predictions_best.csv', index=False)
    
    print(f"✓ 最佳模型预测结果已保存到: rent_final_predictions_best.csv")
    print(f"✓ 预测样本数量: {len(test_predictions)}")
    print(f"✓ 预测租金范围: {test_predictions.min():.2f} - {test_predictions.max():.2f}")
    
except Exception as e:
    print(f"测试集预测出错: {e}")

# 🔥 新增：保存性能表格到CSV
performance_table.to_csv('rent_model_performance_comparison.csv', index=False, encoding='utf-8-sig')
print(f"✓ 租金模型性能对比表格已保存到: rent_model_performance_comparison.csv")

print("\n" + "="*80)
print("🎯 租金预测建模完成!")
print("="*80)
print(f"📊 使用的特征数量: {X_train_array.shape[1]}")
print(f"🏆 最佳模型: {best_model_name}")
print(f"📈 验证集性能:")
print(f"   - RMSE: {results_df.loc[best_model_idx, 'val_rmse']:.4f}")
print(f"   - R²: {results_df.loc[best_model_idx, 'val_r2']:.4f}")
print(f"   - MAE: {results_df.loc[best_model_idx, 'val_mae']:.4f}")
print(f"📊 交叉验证: {results_df.iloc[0]['cv_folds']}折交叉验证R²: {results_df.loc[best_model_idx, 'cv_r2_mean']:.4f}")
print(f"💾 输出文件:")
print(f"   - 最佳模型预测: rent_final_predictions_best.csv")
print(f"   - 线性回归预测: rent_predictions_linear_regression.csv")
print(f"   - Lasso回归预测: rent_predictions_lasso.csv")
print(f"   - 岭回归预测: rent_predictions_ridge.csv")
print(f"   - 弹性网络预测: rent_predictions_elasticnet.csv")
print(f"   - 性能对比表格: rent_model_performance_comparison.csv")
print("="*80)


--- 租金预测建模与评估 ---
1. 数据质量检查和清理...
原始数据形状 - 训练集: (79119, 142), 验证集: (19780, 142), 测试集: (9773, 142)
执行简化数据清理...
清理后 - 训练集NaN: 0
清理后 - 验证集NaN: 0
清理后 - 测试集NaN: 0
清理后形状 - 训练集: (79119, 142), 验证集: (19780, 142), 测试集: (9773, 142)
最终数据形状 - 训练集: (79119, 142), 验证集: (19780, 142), 测试集: (9773, 142)

2. 模型训练和优化...

--- 线性回归 ---
  RMSE: 0.2805, R²: 0.8622
  交叉验证R²: 0.8680 ± 0.0051
  生成线性回归测试集预测...
  ✓ linear_regression预测结果已保存到: rent_predictions_linear_regression.csv
  ✓ 预测样本数量: 9773
  ✓ 预测租金范围: 32559.49 - 7116092.15

--- Lasso回归 ---
  最佳alpha: 0.001
  RMSE: 0.2775, R²: 0.8651
  交叉验证R²: 0.8683 ± 0.0049
  生成Lasso回归测试集预测...
  ✓ lasso预测结果已保存到: rent_predictions_lasso.csv
  ✓ 预测样本数量: 9773
  ✓ 预测租金范围: 65660.08 - 7320069.42

--- 岭回归 ---
  最佳alpha: 10
  RMSE: 0.2768, R²: 0.8658
  交叉验证R²: 0.8687 ± 0.0048
  生成岭回归测试集预测...
  ✓ ridge预测结果已保存到: rent_predictions_ridge.csv
  ✓ 预测样本数量: 9773
  ✓ 预测租金范围: 61336.57 - 7114472.52

--- 弹性网络 ---
  最佳参数: alpha=0.001, l1_ratio=0.1
  RMSE: 0.2762, R²: 0.8664
  交叉验证R²: 0.8687 ± 0.0